In [ ]:
RL_Project_Student_Package_2026\final_submission\industrial_inventory_env\__init__.py


"""Official environment package for the IITM RL inventory-control project."""
from .config import (
    PROJECT_VERSION,
    generate_student_config,
    normalize_roll_number,
    public_config_summary,
    validate_student_config,
)
from .environment import IndustrialInventoryEnv


def make_env(
    roll_number: str,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
) -> IndustrialInventoryEnv:
    """Convenience constructor using a roll number."""
    config = generate_student_config(roll_number)
    return IndustrialInventoryEnv(
        config,
        scenario_mode=scenario_mode,
        domain_randomization=domain_randomization,
    )


__all__ = [
    "IndustrialInventoryEnv",
    "PROJECT_VERSION",
    "generate_student_config",
    "make_env",
    "normalize_roll_number",
    "public_config_summary",
    "validate_student_config",
]


In [ ]:
RL_Project_Student_Package_2026\final_submission\industrial_inventory_env\config.py

"""Deterministic student-variant generation for the 2026 inventory project.

The generator is intentionally deterministic: the same normalized roll number and
project version always map to the same variant.  The catalogue uses balanced
profiles so that aggregate demand, starting inventory and delay exposure remain
similar across variants while product-level values differ.
"""
from __future__ import annotations

from copy import deepcopy
import hashlib
import json
import re
from typing import Any

PROJECT_VERSION = "IITM-6002W-RL-Inventory-2026-v1"

# All profiles remain within the ranges declared in the problem statement.
_DEMAND_PROFILES = [
    [0.90, 1.00, 1.10],
    [0.90, 1.10, 1.00],
    [1.00, 0.90, 1.10],
    [1.00, 1.10, 0.90],
    [1.10, 0.90, 1.00],
    [1.10, 1.00, 0.90],
    [0.95, 1.00, 1.05],
    [0.95, 1.05, 1.00],
    [1.00, 0.95, 1.05],
    [1.00, 1.05, 0.95],
    [1.05, 0.95, 1.00],
    [1.05, 1.00, 0.95],
    [0.85, 1.00, 1.15],
    [0.85, 1.15, 1.00],
    [1.00, 0.85, 1.15],
    [1.00, 1.15, 0.85],
    [1.15, 0.85, 1.00],
    [1.15, 1.00, 0.85],
]

_INITIAL_INVENTORY_PROFILES = [
    [80, 100, 120],
    [80, 120, 100],
    [100, 80, 120],
    [100, 120, 80],
    [120, 80, 100],
    [120, 100, 80],
    [90, 100, 110],
    [90, 110, 100],
    [100, 90, 110],
    [100, 110, 90],
    [110, 90, 100],
    [110, 100, 90],
]

_DELAY_PROFILES = [
    [0.00, 0.05, 0.10],
    [0.00, 0.10, 0.05],
    [0.05, 0.00, 0.10],
    [0.05, 0.10, 0.00],
    [0.10, 0.00, 0.05],
    [0.10, 0.05, 0.00],
    [0.02, 0.05, 0.08],
    [0.02, 0.08, 0.05],
    [0.05, 0.02, 0.08],
    [0.05, 0.08, 0.02],
    [0.08, 0.02, 0.05],
    [0.08, 0.05, 0.02],
]


def _build_variant_catalogue() -> list[dict[str, Any]]:
    """Create a fixed catalogue of balanced variants.

    The catalogue is generated from constant lists, not from runtime randomness,
    so its contents are stable across machines and Python versions.
    """
    catalogue: list[dict[str, Any]] = []
    count = 36
    for index in range(count):
        demand = _DEMAND_PROFILES[index % len(_DEMAND_PROFILES)]
        inventory = _INITIAL_INVENTORY_PROFILES[(index * 5 + 2) % len(_INITIAL_INVENTORY_PROFILES)]
        delay = _DELAY_PROFILES[(index * 7 + 1) % len(_DELAY_PROFILES)]
        catalogue.append(
            {
                "variant_id": f"V{index + 1:03d}",
                "demand_multiplier_profile": list(demand),
                "initial_inventory_profile": list(inventory),
                "lead_time_delay_profile": list(delay),
            }
        )
    return catalogue


VARIANT_CATALOGUE = _build_variant_catalogue()


def normalize_roll_number(roll_number: str) -> str:
    """Normalize and validate a student roll number."""
    if not isinstance(roll_number, str):
        raise TypeError("roll_number must be a string")

    normalized = re.sub(r"\s+", "", roll_number).upper()
    if not normalized or normalized == "ENTER_YOUR_ROLL_NUMBER":
        raise ValueError("Enter your official roll number before generating the configuration.")
    if not re.fullmatch(r"[A-Z0-9_\-/]{4,40}", normalized):
        raise ValueError(
            "Roll number may contain only letters, digits, underscore, hyphen or slash."
        )
    return normalized


def _fingerprint(payload: dict[str, Any]) -> str:
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:16]


def generate_student_config(roll_number: str) -> dict[str, Any]:
    """Generate the official deterministic configuration for a roll number.

    Parameters
    ----------
    roll_number:
        Official student roll number.

    Returns
    -------
    dict
        A JSON-serialisable configuration dictionary.
    """
    normalized = normalize_roll_number(roll_number)
    digest = hashlib.sha256(
        f"{PROJECT_VERSION}:{normalized}".encode("utf-8")
    ).digest()
    variant_index = int.from_bytes(digest[:8], byteorder="big") % len(VARIANT_CATALOGUE)
    variant = deepcopy(VARIANT_CATALOGUE[variant_index])

    config: dict[str, Any] = {
        "project_version": PROJECT_VERSION,
        "roll_number": normalized,
        **variant,
        "declared_ranges": {
            "demand_multiplier": [0.85, 1.15],
            "initial_inventory": [80, 120],
            "lead_time_delay_probability": [0.00, 0.10],
        },
    }
    config["config_fingerprint"] = _fingerprint(config)
    validate_student_config(config)
    return config


def validate_student_config(config: dict[str, Any]) -> None:
    """Validate structure and declared parameter limits."""
    if not isinstance(config, dict):
        raise TypeError("config must be a dictionary")

    required = {
        "project_version",
        "roll_number",
        "variant_id",
        "demand_multiplier_profile",
        "initial_inventory_profile",
        "lead_time_delay_profile",
    }
    missing = required.difference(config)
    if missing:
        raise ValueError(f"Configuration is missing fields: {sorted(missing)}")

    if config["project_version"] != PROJECT_VERSION:
        raise ValueError(
            f"Configuration project version must be {PROJECT_VERSION!r}."
        )

    normalize_roll_number(config["roll_number"])

    demand = config["demand_multiplier_profile"]
    inventory = config["initial_inventory_profile"]
    delays = config["lead_time_delay_profile"]

    if len(demand) != 3 or not all(0.85 <= float(value) <= 1.15 for value in demand):
        raise ValueError("Demand multiplier profile must contain three values in [0.85, 1.15].")
    if len(inventory) != 3 or not all(
        80 <= int(value) <= 120 and int(value) % 10 == 0 for value in inventory
    ):
        raise ValueError(
            "Initial inventory profile must contain three multiples of 10 in [80, 120]."
        )
    if len(delays) != 3 or not all(0.0 <= float(value) <= 0.10 for value in delays):
        raise ValueError("Delay profile must contain three probabilities in [0.00, 0.10].")


def public_config_summary(config: dict[str, Any]) -> dict[str, Any]:
    """Return the fields students should record in their notebook/report."""
    validate_student_config(config)
    keys = [
        "project_version",
        "roll_number",
        "variant_id",
        "config_fingerprint",
        "demand_multiplier_profile",
        "initial_inventory_profile",
        "lead_time_delay_profile",
    ]
    return {key: deepcopy(config[key]) for key in keys}


In [ ]:
RL_Project_Student_Package_2026\final_submission\industrial_inventory_env\environment.py

"""Gymnasium-compatible three-product industrial inventory environment."""
from __future__ import annotations

from copy import deepcopy
from typing import Any, Iterable

import gymnasium as gym
from gymnasium import spaces
import numpy as np

from .config import validate_student_config


class IndustrialInventoryEnv(gym.Env):
    """Multi-product inventory-control environment for the RL course project.

    Internal actions are indices in ``MultiDiscrete([11, 11, 11])``.  Index ``a_i``
    represents an order quantity of ``10 * a_i`` units for product ``i``.

    Parameters
    ----------
    student_config:
        Configuration returned by ``generate_student_config``.
    scenario_mode:
        One of ``"random"``, ``"stationary"``, ``"seasonal"``, ``"trend"``,
        ``"shock"`` or ``"mixed"``.  A sequence such as ``["seasonal", "trend"]``
        is also accepted.
    domain_randomization:
        When True, each episode applies small bounded perturbations around the
        assigned student profile.  All realised values remain within the ranges
        declared in the problem statement.
    """

    metadata = {"render_modes": []}

    NUM_PRODUCTS = 3
    CAPACITY = 1000.0
    PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float64)
    HOLDING_COST_PER_VOLUME = 5.0
    STOCKOUT_COSTS = np.asarray([400.0, 500.0, 300.0], dtype=np.float64)
    FIXED_ORDERING_COSTS = np.asarray([80.0, 200.0, 120.0], dtype=np.float64)
    DISCARDING_COSTS = np.asarray([200.0, 250.0, 150.0], dtype=np.float64)
    REFERENCE_LEAD_TIMES = np.asarray([3, 2, 1], dtype=np.int64)
    REFERENCE_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float64)
    HORIZON = 50
    PIPELINE_DAYS = 4
    DEMAND_HISTORY_DAYS = 7

    _VALID_SCENARIO_COMPONENTS = {"seasonal", "trend", "shock"}

    def __init__(
        self,
        student_config: dict[str, Any],
        scenario_mode: str | Iterable[str] = "random",
        domain_randomization: bool = True,
    ) -> None:
        super().__init__()
        validate_student_config(student_config)
        self.student_config = deepcopy(student_config)
        self.scenario_mode = scenario_mode
        self.domain_randomization = bool(domain_randomization)

        self.action_space = spaces.MultiDiscrete(
            np.asarray([11, 11, 11], dtype=np.int64)
        )
        self.observation_space = spaces.Dict(
            {
                "inventory": spaces.Box(
                    low=0,
                    high=1000,
                    shape=(3,),
                    dtype=np.int32,
                ),
                "arrival_pipeline": spaces.Box(
                    low=0,
                    high=10000,
                    shape=(3, 4),
                    dtype=np.int32,
                ),
                "demand_history": spaces.Box(
                    low=0,
                    high=10000,
                    shape=(7, 3),
                    dtype=np.int32,
                ),
                "day": spaces.Box(
                    low=0,
                    high=self.HORIZON,
                    shape=(1,),
                    dtype=np.int32,
                ),
                "capacity_utilisation": spaces.Box(
                    low=0.0,
                    high=1.0,
                    shape=(1,),
                    dtype=np.float32,
                ),
            }
        )

        self.day = 0
        self.inventory = np.zeros(3, dtype=np.int64)
        self.arrival_pipeline = np.zeros((3, 4), dtype=np.int64)
        self.demand_history = np.zeros((7, 3), dtype=np.int64)
        self.episode_cost = 0.0
        self.episode_parameters: dict[str, Any] = {}
        self._demand_sequence = np.zeros((self.HORIZON, 3), dtype=np.int64)
        self._delay_flags = np.zeros((self.HORIZON, 3), dtype=bool)

    @staticmethod
    def quantities_to_action_indices(quantities: Iterable[int]) -> np.ndarray:
        """Convert actual order quantities to internal action indices."""
        array = np.asarray(list(quantities))
        if array.shape != (3,):
            raise ValueError("Order quantities must contain exactly three values.")
        if not np.issubdtype(array.dtype, np.number):
            raise TypeError("Order quantities must be numeric.")
        if not np.all(np.isfinite(array)):
            raise ValueError("Order quantities must be finite.")
        if not np.all(array == np.round(array)):
            raise ValueError("Order quantities must be integers.")
        array = array.astype(np.int64)
        if not np.all((array >= 0) & (array <= 100) & (array % 10 == 0)):
            raise ValueError("Each order quantity must be one of 0, 10, ..., 100.")
        return array // 10

    @staticmethod
    def action_indices_to_quantities(
        action: Iterable[int],
    ) -> np.ndarray:
        """Convert internal action indices to order quantities."""

        array = np.asarray(list(action))

        if array.shape != (3,):
            raise ValueError(
                "Action must contain exactly three values."
            )

        if not np.issubdtype(array.dtype, np.number):
            raise TypeError("Action indices must be numeric.")

        if not np.all(np.isfinite(array)):
            raise ValueError("Action indices must be finite.")

        if not np.all(array == np.round(array)):
            raise ValueError("Action indices must be integers.")

        array = array.astype(np.int64)

        if not np.all((array >= 0) & (array <= 10)):
            raise ValueError(
                "Action must contain three integer indices in [0, 10]."
            )

        return array * 10

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict[str, Any] | None = None,
    ) -> tuple[dict[str, np.ndarray], dict[str, Any]]:
        super().reset(seed=seed)
        options = options or {}

        self.day = 0
        self.episode_cost = 0.0
        self.arrival_pipeline = np.zeros((3, 4), dtype=np.int64)
        self.demand_history = np.zeros((7, 3), dtype=np.int64)

        self.episode_parameters = self._sample_episode_parameters(options)
        self.inventory = np.asarray(
            self.episode_parameters["initial_inventory"], dtype=np.int64
        ).copy()
        self._prepare_episode_sequences()

        observation = self._get_observation()
        info = self._reset_info(seed)
        return observation, info

    def step(
        self, action: np.ndarray | list[int] | tuple[int, int, int]
    ) -> tuple[dict[str, np.ndarray], float, bool, bool, dict[str, Any]]:
        if self.day >= self.HORIZON:
            raise RuntimeError("Episode is complete. Call reset() before step().")

        action_array = np.asarray(action, dtype=np.int64)
        if not self.action_space.contains(action_array):
            raise ValueError(
                "Environment action must contain three integer indices in [0, 10]. "
                "Use quantities_to_action_indices() when starting from actual quantities."
            )
        order_quantities = action_array * 10

        # 1. Receive orders due today, then advance the future-arrival pipeline.
        scheduled_arrivals = self.arrival_pipeline[:, 0].copy()
        self.arrival_pipeline[:, :-1] = self.arrival_pipeline[:, 1:]
        self.arrival_pipeline[:, -1] = 0

        # 2. Enforce warehouse capacity after arrivals.
        accepted_arrivals, discarded = self._accept_arrivals_with_capacity(
            scheduled_arrivals
        )
        self.inventory += accepted_arrivals

        # 3. Add today's order to its future arrival position.
        realised_delays = self._delay_flags[self.day].astype(np.int64)
        for product in range(self.NUM_PRODUCTS):
            quantity = int(order_quantities[product])
            if quantity == 0:
                continue
            lead_time = int(self.REFERENCE_LEAD_TIMES[product] + realised_delays[product])
            pipeline_index = lead_time - 1
            self.arrival_pipeline[product, pipeline_index] += quantity

        # 4-5. Generate and serve demand.
        demand = self._demand_sequence[self.day].copy()
        fulfilled = np.minimum(self.inventory, demand)
        unfulfilled = demand - fulfilled
        self.inventory -= fulfilled

        # 6. Update history and state summaries.
        self.demand_history[:-1] = self.demand_history[1:]
        self.demand_history[-1] = demand

        # 7. Calculate official costs and reward.
        holding_cost = float(
            np.dot(self.inventory, self.PRODUCT_VOLUMES)
            * self.HOLDING_COST_PER_VOLUME
        )
        stockout_cost = float(np.dot(unfulfilled, self.STOCKOUT_COSTS))
        ordering_cost = float(
            np.dot(order_quantities > 0, self.FIXED_ORDERING_COSTS)
        )
        discard_cost = float(np.dot(discarded, self.DISCARDING_COSTS))
        daily_cost = holding_cost + stockout_cost + ordering_cost + discard_cost
        reward = -daily_cost / 100.0
        self.episode_cost += daily_cost

        current_day = self.day
        self.day += 1
        terminated = False
        truncated = self.day >= self.HORIZON

        observation = self._get_observation()
        info = {
            "day": current_day,
            "demand": demand.astype(np.int32),
            "fulfilled_demand": fulfilled.astype(np.int32),
            "unfulfilled_demand": unfulfilled.astype(np.int32),
            "scheduled_arrivals": scheduled_arrivals.astype(np.int32),
            "accepted_arrivals": accepted_arrivals.astype(np.int32),
            "discarded_units": discarded.astype(np.int32),
            "order_quantities": order_quantities.astype(np.int32),
            "realised_one_day_delay": realised_delays.astype(np.int32),
            "costs": {
                "holding": holding_cost,
                "stockout": stockout_cost,
                "ordering": ordering_cost,
                "discarding": discard_cost,
                "daily_total": daily_cost,
                "episode_total": self.episode_cost,
            },
        }
        return observation, float(reward), terminated, truncated, info

    def _sample_episode_parameters(self, options: dict[str, Any]) -> dict[str, Any]:
        base_demand = np.asarray(
            self.student_config["demand_multiplier_profile"], dtype=np.float64
        )
        base_inventory = np.asarray(
            self.student_config["initial_inventory_profile"], dtype=np.int64
        )
        base_delay = np.asarray(
            self.student_config["lead_time_delay_profile"], dtype=np.float64
        )

        if self.domain_randomization:
            demand_offsets = self.np_random.choice(
                np.asarray([-0.05, 0.0, 0.05]), size=3
            )
            demand_multipliers = np.clip(
                base_demand + demand_offsets, 0.85, 1.15
            )

            inventory_offsets = self.np_random.choice(
                np.asarray([-10, 0, 10]), size=3
            )
            initial_inventory = np.clip(
                base_inventory + inventory_offsets, 80, 120
            ).astype(np.int64)

            delay_offsets = self.np_random.choice(
                np.asarray([-0.02, 0.0, 0.02]), size=3
            )
            delay_probabilities = np.clip(
                base_delay + delay_offsets, 0.0, 0.10
            )
        else:
            demand_multipliers = base_demand.copy()
            initial_inventory = base_inventory.copy()
            delay_probabilities = base_delay.copy()

        scenario_components = self._resolve_scenario_components(
            options.get("scenario_mode", self.scenario_mode)
        )

        return {
            "variant_id": self.student_config["variant_id"],
            "demand_multipliers": demand_multipliers.tolist(),
            "initial_inventory": initial_inventory.tolist(),
            "delay_probabilities": delay_probabilities.tolist(),
            "scenario_components": scenario_components,
        }

    def _resolve_scenario_components(
        self, mode: str | Iterable[str]
    ) -> list[str]:
        if isinstance(mode, str):
            normalized = mode.strip().lower()
            if normalized == "stationary":
                return []
            if normalized in self._VALID_SCENARIO_COMPONENTS:
                return [normalized]
            if normalized == "mixed":
                count = 2
                return sorted(
                    self.np_random.choice(
                        sorted(self._VALID_SCENARIO_COMPONENTS),
                        size=count,
                        replace=False,
                    ).tolist()
                )
            if normalized == "random":
                count = int(self.np_random.choice([0, 1, 2], p=[0.25, 0.50, 0.25]))
                if count == 0:
                    return []
                return sorted(
                    self.np_random.choice(
                        sorted(self._VALID_SCENARIO_COMPONENTS),
                        size=count,
                        replace=False,
                    ).tolist()
                )
            raise ValueError(
                "scenario_mode must be random, stationary, seasonal, trend, shock or mixed."
            )

        components = sorted({str(item).strip().lower() for item in mode})
        invalid = set(components).difference(self._VALID_SCENARIO_COMPONENTS)
        if invalid:
            raise ValueError(f"Unsupported scenario components: {sorted(invalid)}")
        return components

    def _prepare_episode_sequences(self) -> None:
        multipliers = np.asarray(
            self.episode_parameters["demand_multipliers"], dtype=np.float64
        )
        expected = np.tile(
            self.REFERENCE_DEMAND_MEANS * multipliers,
            (self.HORIZON, 1),
        )
        days = np.arange(self.HORIZON, dtype=np.float64)
        components = set(self.episode_parameters["scenario_components"])
        scenario_details: dict[str, Any] = {}

        if "seasonal" in components:
            amplitudes = self.np_random.uniform(0.05, 0.12, size=3)
            phases = self.np_random.uniform(0.0, 2.0 * np.pi, size=3)
            seasonal_factor = 1.0 + amplitudes[None, :] * np.sin(
                2.0 * np.pi * days[:, None] / 7.0 + phases[None, :]
            )
            expected *= seasonal_factor
            scenario_details["seasonal_amplitudes"] = amplitudes.tolist()

        if "trend" in components:
            direction = int(self.np_random.choice([-1, 1]))
            magnitude = float(self.np_random.uniform(0.10, 0.22))
            centered = np.linspace(-0.5, 0.5, self.HORIZON)
            trend_factor = 1.0 + direction * magnitude * centered
            expected *= trend_factor[:, None]
            scenario_details["trend_direction"] = "up" if direction > 0 else "down"
            scenario_details["trend_total_change"] = magnitude

        if "shock" in components:
            product = int(self.np_random.integers(0, self.NUM_PRODUCTS))
            start = int(self.np_random.integers(8, 36))
            duration = int(self.np_random.integers(3, 8))
            factor = float(self.np_random.uniform(1.20, 1.45))
            end = min(start + duration, self.HORIZON)
            expected[start:end, product] *= factor
            scenario_details["shock_product"] = product + 1
            scenario_details["shock_start_day"] = start
            scenario_details["shock_duration"] = end - start
            scenario_details["shock_factor"] = factor

        expected = np.maximum(expected, 0.01)
        self._demand_sequence = self.np_random.poisson(expected).astype(np.int64)
        delay_probabilities = np.asarray(
            self.episode_parameters["delay_probabilities"], dtype=np.float64
        )
        self._delay_flags = (
            self.np_random.random((self.HORIZON, self.NUM_PRODUCTS))
            < delay_probabilities[None, :]
        )
        self.episode_parameters["scenario_details"] = scenario_details

    def _accept_arrivals_with_capacity(
        self, arrivals: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        arrivals = np.asarray(arrivals, dtype=np.int64)
        current_volume = float(np.dot(self.inventory, self.PRODUCT_VOLUMES))
        available_volume = max(self.CAPACITY - current_volume, 0.0)
        arrival_volume = float(np.dot(arrivals, self.PRODUCT_VOLUMES))

        if arrival_volume <= available_volume + 1e-9:
            return arrivals.copy(), np.zeros(3, dtype=np.int64)
        if available_volume <= 0.0:
            return np.zeros(3, dtype=np.int64), arrivals.copy()

        fraction = available_volume / arrival_volume
        raw_acceptance = arrivals.astype(np.float64) * fraction
        accepted = np.floor(raw_acceptance).astype(np.int64)
        used_volume = float(np.dot(accepted, self.PRODUCT_VOLUMES))
        remaining_volume = max(available_volume - used_volume, 0.0)

        fractional_parts = raw_acceptance - accepted
        priority = sorted(
            range(self.NUM_PRODUCTS),
            key=lambda index: (-fractional_parts[index], index),
        )
        made_progress = True
        while made_progress:
            made_progress = False
            for product in priority:
                if accepted[product] >= arrivals[product]:
                    continue
                volume = float(self.PRODUCT_VOLUMES[product])
                if volume <= remaining_volume + 1e-9:
                    accepted[product] += 1
                    remaining_volume -= volume
                    made_progress = True

        discarded = arrivals - accepted
        return accepted, discarded

    def _get_observation(self) -> dict[str, np.ndarray]:
        utilisation = float(
            np.dot(self.inventory, self.PRODUCT_VOLUMES) / self.CAPACITY
        )
        return {
            "inventory": self.inventory.astype(np.int32).copy(),
            "arrival_pipeline": self.arrival_pipeline.astype(np.int32).copy(),
            "demand_history": self.demand_history.astype(np.int32).copy(),
            "day": np.asarray([self.day], dtype=np.int32),
            "capacity_utilisation": np.asarray([utilisation], dtype=np.float32),
        }

    def _reset_info(self, seed: int | None) -> dict[str, Any]:
        return {
            "seed": seed,
            "roll_number": self.student_config["roll_number"],
            "variant_id": self.student_config["variant_id"],
            "config_fingerprint": self.student_config.get("config_fingerprint"),
            "episode_parameters": deepcopy(self.episode_parameters),
        }


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\a2c\metadata.json

{
  "technique_category": "Advantage Actor-Critic (A2C)",
  "policy_file": "policy.py",
  "model_artifact": "model.zip",
  "observation_representation": "35-dim hand-engineered features (training_utils/obs_wrapper.py::flatten_observation); joint action encoded as MultiDiscrete([11, 11, 11]) over quantities {0, 10, ..., 100} per product",
  "reward_used_during_training": "Base environment reward (negative scaled daily cost) plus an auxiliary ShapedReward wrapper (extra stockout penalty, discard/utilisation penalty, order-batching nudge) annealed linearly to zero over the training budget. All reported cost/service metrics use the raw, unshaped environment reward -- shaping never affects evaluation or this frozen policy.",
  "main_hyperparameters": {
    "learning_rate": 0.0007,
    "n_steps": 32,
    "gae_lambda": 1.0,
    "ent_coef": 0.01,
    "net_arch": [128, 128],
    "total_timesteps": 200000
  },
  "training_pipeline_scripts": ["training_pipelines/training_scripts/train_a2c.py"],
  "notes": "A later 12-configuration screen + 3-seed promotion refinement attempt did not beat this original iteration-1 model, so the original was kept.",
  "local_validation_performance": {
    "suite": "A (assigned configuration, 40 seeds x 5 scenario modes = 200 episodes)",
    "mean_cost": 114669.1,
    "mean_service_level": 0.969282,
    "minimum_scenario_service_level": 0.963480
  },
  "leaderboard_submission_id": null
}


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\a2c\policy.py

"""A2C submission for the IITM RL Inventory Control project.

Technique: Advantage Actor-Critic (A2C)
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
from stable_baselines3 import A2C

_PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float32)
_CAPACITY = 1000.0
_REF_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float32)


def _flatten_observation(observation) -> np.ndarray:
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(
        np.asarray(observation["capacity_utilisation"]).reshape(-1)[0]
    )

    last3_mean = demand_history[-3:].mean(axis=0)
    last7_mean = demand_history.mean(axis=0)
    last7_std = demand_history.std(axis=0)

    inv_position_units = inventory + pipeline.sum(axis=1)
    inv_position_volume = inv_position_units * _PRODUCT_VOLUMES

    return np.concatenate(
        [
            inventory / 100.0,
            pipeline.reshape(-1) / 100.0,
            last3_mean / _REF_DEMAND_MEANS,
            last7_mean / _REF_DEMAND_MEANS,
            last7_std / _REF_DEMAND_MEANS,
            np.asarray([day / 50.0], dtype=np.float32),
            np.asarray([capacity_utilisation], dtype=np.float32),
            inv_position_units / 200.0,
            inv_position_volume / _CAPACITY,
            (inv_position_units - last3_mean) / 100.0,
        ],
        dtype=np.float32,
    ).astype(np.float32, copy=False)


_MODEL_PATH = Path(__file__).resolve().parent / "model.zip"
_MODEL = A2C.load(str(_MODEL_PATH), device="cpu")


def run_policy(observation):
    features = _flatten_observation(observation)
    action, _ = _MODEL.predict(features, deterministic=True)
    action = np.asarray(action, dtype=np.int64).reshape(-1)
    quantities = (action[:3] * 10).tolist()
    return [int(q) for q in quantities]


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\dqn\metadata.json

{
  "technique_category": "Deep Q-Network (DQN)",
  "policy_file": "policy.py",
  "model_artifact": "model.zip",
  "observation_representation": "35-dim hand-engineered features (training_utils/obs_wrapper.py::flatten_observation); joint 1331-action space encoded as a single Discrete index via base-11 encoding (action_wrapper.py), decoded to quantities {0, 10, ..., 100} per product at inference time",
  "reward_used_during_training": "Base environment reward (negative scaled daily cost) plus an auxiliary ShapedReward wrapper (extra stockout penalty, discard/utilisation penalty, order-batching nudge) annealed linearly to zero over the training budget. All reported cost/service metrics use the raw, unshaped environment reward -- shaping never affects evaluation or this frozen policy.",
  "main_hyperparameters": {
    "learning_rate": 0.0005,
    "buffer_size": 200000,
    "target_update_interval": 2000,
    "exploration_fraction": 0.3,
    "net_arch": [256, 256],
    "total_timesteps": 150000
  },
  "training_pipeline_scripts": ["training_pipelines/training_scripts/train_dqn.py"],
  "local_validation_performance": {
    "suite": "A (assigned configuration, 40 seeds x 5 scenario modes = 200 episodes)",
    "mean_cost": 96871.3,
    "mean_service_level": 0.988895,
    "minimum_scenario_service_level": 0.988213
  },
  "leaderboard_submission_id": null
}


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\dqn\policy.py

"""DQN submission for the IITM RL Inventory Control project.

Technique: Deep Q-Network (DQN)

Trained with a flattened Discrete(1331) action space (base-11 encoding of the
underlying MultiDiscrete([11,11,11]) space). Decoded here into three
per-product order quantities.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
from stable_baselines3 import DQN

_PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float32)
_CAPACITY = 1000.0
_REF_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float32)


def _flatten_observation(observation) -> np.ndarray:
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(
        np.asarray(observation["capacity_utilisation"]).reshape(-1)[0]
    )

    last3_mean = demand_history[-3:].mean(axis=0)
    last7_mean = demand_history.mean(axis=0)
    last7_std = demand_history.std(axis=0)

    inv_position_units = inventory + pipeline.sum(axis=1)
    inv_position_volume = inv_position_units * _PRODUCT_VOLUMES

    return np.concatenate(
        [
            inventory / 100.0,
            pipeline.reshape(-1) / 100.0,
            last3_mean / _REF_DEMAND_MEANS,
            last7_mean / _REF_DEMAND_MEANS,
            last7_std / _REF_DEMAND_MEANS,
            np.asarray([day / 50.0], dtype=np.float32),
            np.asarray([capacity_utilisation], dtype=np.float32),
            inv_position_units / 200.0,
            inv_position_volume / _CAPACITY,
            (inv_position_units - last3_mean) / 100.0,
        ],
        dtype=np.float32,
    ).astype(np.float32, copy=False)


def _decode_joint_index(index: int) -> list[int]:
    """Base-11 decode into three order quantities in {0, 10, ..., 100}."""
    index = int(index)
    a2 = index % 11
    index //= 11
    a1 = index % 11
    a0 = index // 11
    return [a0 * 10, a1 * 10, a2 * 10]


_MODEL_PATH = Path(__file__).resolve().parent / "model.zip"
_MODEL = DQN.load(str(_MODEL_PATH), device="cpu")


def run_policy(observation):
    features = _flatten_observation(observation)
    action, _ = _MODEL.predict(features, deterministic=True)
    joint_index = int(np.asarray(action).reshape(-1)[0])
    return _decode_joint_index(joint_index)


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\neural_sarsa\policy.py


"""Neural Network SARSA submission (v3, iteration-3 "RL Final Run" doc,
Phase 5 -- true SARSA target, replay buffer stores the ACTUAL next action).

Technique: Neural Network SARSA (joint 1331-action Q-network), config
sarsa_c (learning_rate=3e-4, gamma=0.98, epsilon_final=0.03), promoted seed
20260825, 500,000 transitions. Uses Representation B (raw + engineered,
76-dim -- FIXED zero-padding + days-of-supply formulas vs the v1 submission).

At inference time the SARSA target collapses to plain greedy argmax over
Q(s, .), decoded from the joint action index via the same base-11 encoding
used by the DQN-family submissions.

The observation-flattening code is inlined so this submission does not
depend on the local `src` package.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

_PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float32)
_CAPACITY = 1000.0
_HORIZON = 50.0
_REFERENCE_LEAD_TIMES = np.asarray([3.0, 2.0, 1.0], dtype=np.float32)
_DEMAND_HISTORY_DAYS = 7

_INVENTORY_SCALE = 200.0
_PIPELINE_SCALE = 100.0
_DEMAND_HISTORY_SCALE = 100.0
_DAY_SCALE = 49.0


def _flatten_raw(observation) -> np.ndarray:
    """Representation A: 38 raw, normalized features."""
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(
        np.asarray(observation["capacity_utilisation"]).reshape(-1)[0]
    )
    return np.concatenate(
        [
            inventory / _INVENTORY_SCALE,
            pipeline.reshape(-1) / _PIPELINE_SCALE,
            demand_history.reshape(-1) / _DEMAND_HISTORY_SCALE,
            np.asarray([day / _DAY_SCALE], dtype=np.float32),
            np.asarray([capacity_utilisation], dtype=np.float32),
        ]
    ).astype(np.float32, copy=False)


def _flatten_engineered(observation) -> np.ndarray:
    """38 engineered features: 30 per-product + 8 global."""
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = int(np.asarray(observation["day"]).reshape(-1)[0])

    inventory_position = inventory + pipeline.sum(axis=1)

    # `demand_history` is zero-padded at the start of an episode -- only the
    # last `valid_days` rows are real observations; ignore the padding.
    valid_days = min(day, _DEMAND_HISTORY_DAYS)
    if valid_days == 0:
        last3_mean = np.zeros(3, dtype=np.float32)
        last7_mean = np.zeros(3, dtype=np.float32)
        demand_std = np.zeros(3, dtype=np.float32)
    else:
        valid_history = demand_history[-valid_days:]
        valid3 = valid_history[-min(valid_days, 3):]
        last3_mean = valid3.mean(axis=0)
        last7_mean = valid_history.mean(axis=0)
        demand_std = valid_history.std(axis=0)
    demand_trend = last3_mean - last7_mean

    within_lead_time = np.zeros(3, dtype=np.float32)
    after_lead_time = np.zeros(3, dtype=np.float32)
    for product in range(3):
        lead_time = int(_REFERENCE_LEAD_TIMES[product])
        within_lead_time[product] = pipeline[product, :lead_time].sum()
        after_lead_time[product] = pipeline[product, lead_time:].sum()

    lead_time_demand_estimate = last7_mean * _REFERENCE_LEAD_TIMES
    inventory_position_gap = inventory_position - lead_time_demand_estimate
    estimated_days_of_supply = inventory_position / np.maximum(last7_mean, 1.0)

    per_product = np.stack(
        [
            inventory_position, last3_mean, last7_mean, demand_trend, demand_std,
            within_lead_time, after_lead_time, lead_time_demand_estimate,
            inventory_position_gap, estimated_days_of_supply,
        ],
        axis=1,
    ).reshape(-1).astype(np.float32)

    current_volume = float(np.dot(inventory, _PRODUCT_VOLUMES))
    pipeline_volume = float(np.dot(pipeline.sum(axis=1), _PRODUCT_VOLUMES))
    current_volume_ratio = current_volume / _CAPACITY
    headroom_ratio = max(_CAPACITY - current_volume, 0.0) / _CAPACITY
    pipeline_volume_ratio = pipeline_volume / _CAPACITY
    projected_utilisation = (current_volume + pipeline_volume) / _CAPACITY
    remaining_fraction = max(_HORIZON - day, 0.0) / _HORIZON
    phase = day / _HORIZON
    early = 1.0 if phase < (1.0 / 3.0) else 0.0
    middle = 1.0 if (1.0 / 3.0) <= phase < (2.0 / 3.0) else 0.0
    late = 1.0 if phase >= (2.0 / 3.0) else 0.0
    global_features = np.asarray(
        [current_volume_ratio, headroom_ratio, pipeline_volume_ratio, projected_utilisation,
         remaining_fraction, early, middle, late],
        dtype=np.float32,
    )
    return np.concatenate([per_product, global_features]).astype(np.float32)


def _flatten_observation(observation) -> np.ndarray:
    """Representation B: raw (38) + engineered (38) = 76 features."""
    return np.concatenate(
        [_flatten_raw(observation), _flatten_engineered(observation)]
    ).astype(np.float32)


def _decode_joint_index(index: int) -> list[int]:
    """Base-11 decode into three order quantities in {0, 10, ..., 100}
    (joint_index = a1*121 + a2*11 + a3, see src/environment/action_codec.py)."""
    index = int(index)
    a3 = index % 11
    index //= 11
    a2 = index % 11
    a1 = index // 11
    return [a1 * 10, a2 * 10, a3 * 10]


class _QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden_sizes) -> None:
        super().__init__()
        layers = []
        last = obs_dim
        for size in hidden_sizes:
            layers.append(nn.Linear(last, size))
            layers.append(nn.ReLU())
            last = size
        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(last, n_actions)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.head(self.trunk(obs))


_CKPT_PATH = Path(__file__).resolve().parent / "policy_state.pt"
_CKPT = torch.load(str(_CKPT_PATH), map_location="cpu", weights_only=False)

_MODEL = _QNetwork(int(_CKPT["obs_dim"]), int(_CKPT["n_actions"]), list(_CKPT["hidden_sizes"]))
_MODEL.load_state_dict(_CKPT["model_state"])
_MODEL.eval()


def run_policy(observation):
    """Deterministic (greedy) Neural SARSA inference producing three order quantities."""
    features = _flatten_observation(observation)
    with torch.no_grad():
        q_values = _MODEL(torch.from_numpy(features).unsqueeze(0))
    joint_index = int(torch.argmax(q_values, dim=-1).item())
    return _decode_joint_index(joint_index)


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\neural_sarsa\metadata.json

{
  "technique_category": "Neural Network SARSA",
  "policy_file": "policy.py",
  "model_artifact": "policy_state.pt",
  "observation_representation": "76-dim Representation B (src/features/observation.py + engineered.py: 38 raw normalised features + 38 further-engineered features -- inventory-position gap, lead-time-demand estimate, days-of-supply, demand trend, capacity/horizon features); joint 1331-action space via base-11 encoding, decoded to quantities {0, 10, ..., 100} per product at inference time",
  "reward_used_during_training": "Base environment reward (negative scaled daily cost) plus an auxiliary ShapedReward wrapper (extra stockout penalty, discard/utilisation penalty, order-batching nudge) annealed linearly to zero over the training budget. All reported cost/service metrics use the raw, unshaped environment reward -- shaping never affects evaluation or this frozen policy.",
  "main_hyperparameters": {
    "algorithm_note": "True SARSA target -- the TD target evaluates Q at the action the epsilon-greedy behaviour policy actually took, never argmax",
    "learning_rate": 0.0003,
    "gamma": 0.98,
    "epsilon_start": 1.0,
    "epsilon_end": 0.03,
    "config_id": "sarsa_c",
    "promoted_seed": 20260825,
    "total_transitions": 500000
  },
  "training_pipeline_scripts": [
    "training_pipelines/run_neural_sarsa_screening_v2.py",
    "training_pipelines/run_neural_sarsa_promote_v2.py"
  ],
  "local_validation_performance": {
    "suite": "A (assigned configuration, 40 seeds x 5 scenario modes = 200 episodes)",
    "mean_cost": 111730.6,
    "mean_service_level": 0.992882,
    "minimum_scenario_service_level": 0.991478
  },
  "leaderboard_submission_id": null
}


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\ppo\policy.py


"""PPO submission for the IITM RL Inventory Control project.

Technique: Proximal Policy Optimization (PPO)

Loads a Stable-Baselines3 PPO model at import time and performs deterministic
inference in `run_policy`. The observation-flattening code is inlined so that
the submission does not depend on any local training package.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
from stable_baselines3 import PPO

# ---------------------------------------------------------------------------
# Constants mirrored from the environment.
# ---------------------------------------------------------------------------
_PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float32)
_CAPACITY = 1000.0
_REF_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float32)


def _flatten_observation(observation) -> np.ndarray:
    """Reproduce the training-time observation-flattening exactly."""
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(
        np.asarray(observation["capacity_utilisation"]).reshape(-1)[0]
    )

    last3_mean = demand_history[-3:].mean(axis=0)
    last7_mean = demand_history.mean(axis=0)
    last7_std = demand_history.std(axis=0)

    inv_position_units = inventory + pipeline.sum(axis=1)
    inv_position_volume = inv_position_units * _PRODUCT_VOLUMES

    return np.concatenate(
        [
            inventory / 100.0,
            pipeline.reshape(-1) / 100.0,
            last3_mean / _REF_DEMAND_MEANS,
            last7_mean / _REF_DEMAND_MEANS,
            last7_std / _REF_DEMAND_MEANS,
            np.asarray([day / 50.0], dtype=np.float32),
            np.asarray([capacity_utilisation], dtype=np.float32),
            inv_position_units / 200.0,
            inv_position_volume / _CAPACITY,
            (inv_position_units - last3_mean) / 100.0,
        ],
        dtype=np.float32,
    ).astype(np.float32, copy=False)


# ---------------------------------------------------------------------------
# Load the model once at import time.
# ---------------------------------------------------------------------------
_MODEL_PATH = Path(__file__).resolve().parent / "model.zip"
_MODEL = PPO.load(str(_MODEL_PATH), device="cpu")


def run_policy(observation):
    """Deterministic PPO inference producing three order quantities."""
    features = _flatten_observation(observation)
    action, _ = _MODEL.predict(features, deterministic=True)
    action = np.asarray(action, dtype=np.int64).reshape(-1)
    quantities = (action[:3] * 10).tolist()
    return [int(q) for q in quantities]


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\ppo\metadata.json

{
  "technique_category": "Proximal Policy Optimization (PPO)",
  "policy_file": "policy.py",
  "model_artifact": "model.zip",
  "observation_representation": "35-dim hand-engineered features (training_utils/obs_wrapper.py::flatten_observation); joint action encoded as MultiDiscrete([11, 11, 11]) over quantities {0, 10, ..., 100} per product",
  "reward_used_during_training": "Base environment reward (negative scaled daily cost) plus an auxiliary ShapedReward wrapper (extra stockout penalty, discard/utilisation penalty, order-batching nudge) annealed linearly to zero over the training budget. All reported cost/service metrics use the raw, unshaped environment reward -- shaping never affects evaluation or this frozen policy.",
  "main_hyperparameters": {
    "learning_rate": 0.0003,
    "ent_coef": 0.0,
    "n_steps": 512,
    "batch_size": 512,
    "gamma": 0.99,
    "net_arch": [128, 128],
    "total_timesteps": 1500000
  },
  "training_pipeline_scripts": ["training_pipelines/run_ppo_screening.py", "training_pipelines/run_ppo_promote.py"],
  "local_validation_performance": {
    "suite": "A (assigned configuration, 40 seeds x 5 scenario modes = 200 episodes)",
    "mean_cost": 91469.9,
    "mean_service_level": 0.987641,
    "minimum_scenario_service_level": 0.985131
  },
  "leaderboard_submission_id": null
}


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\td_lambda\metadata.json

{
  "technique_category": "TD(lambda) with Eligibility Traces",
  "policy_file": "policy.py",
  "model_artifact": "policy_weights.npz",
  "observation_representation": "54-dim sparse coarse-coded 'band' discretisation (src/algorithms/common/discretizer.py), purpose-built for a linear function approximator; reduced 48-action catalogue (src/algorithms/common/action_catalogue.py) mapping directly to quantities in {0, 20, ..., 100} per product",
  "reward_used_during_training": "Base environment reward (negative scaled daily cost) plus an auxiliary ShapedReward wrapper (extra stockout penalty, discard/utilisation penalty, order-batching nudge) annealed linearly to zero over the training budget. All reported cost/service metrics use the raw, unshaped environment reward -- shaping never affects evaluation or this frozen policy.",
  "main_hyperparameters": {
    "algorithm_note": "True Online SARSA(lambda), linear function approximation, dutch eligibility traces",
    "gamma": 0.98,
    "lambda": 0.7,
    "alpha": 0.01,
    "epsilon_start": 0.4,
    "epsilon_end": 0.02,
    "total_episodes": 5000,
    "seeds_used": 3
  },
  "training_pipeline_scripts": [
    "training_pipelines/run_td_lambda_grid_search.py",
    "training_pipelines/run_td_lambda_three_seeds.py"
  ],
  "local_validation_performance": {
    "suite": "A (assigned configuration, 40 seeds x 5 scenario modes = 200 episodes)",
    "mean_cost": 122769.5,
    "mean_service_level": 0.989016,
    "minimum_scenario_service_level": 0.987600
  },
  "leaderboard_submission_id": null
}


In [ ]:
RL_Project_Student_Package_2026\final_submission\submissions\td_lambda\policy.py

"""TD(lambda) submission for the IITM RL Inventory Control project.

Technique: True Online SARSA(lambda), linear function approximation over
sparse coarse-coded ("banded") features, with a reduced 48-action catalogue.

Self-contained: inlines the band discretizer and the action catalogue so
this submission does not depend on the local `src` package.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np

_REFERENCE_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float32)
_REFERENCE_LEAD_TIMES = np.asarray([3.0, 2.0, 1.0], dtype=np.float32)
_PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float32)
_CAPACITY = 1000.0
_HORIZON = 50.0
_EPS = 1e-6

N_BINS = 3
N_BAND_GROUPS = 18
FEATURE_DIM = N_BAND_GROUPS * N_BINS  # 54

# 48-action reduced catalogue (doc section 8.2) -- must match
# src/algorithms/common/action_catalogue.py's ACTION_CATALOGUE exactly.
ACTION_CATALOGUE = [
    (0, 0, 0), (0, 0, 20), (0, 0, 40), (0, 0, 60), (0, 0, 80), (0, 0, 100),
    (0, 20, 0), (0, 40, 0), (0, 40, 40), (0, 60, 0), (0, 60, 60), (0, 80, 0),
    (0, 80, 80), (0, 100, 0), (0, 100, 100), (20, 0, 0), (20, 20, 20),
    (20, 40, 40), (20, 60, 60), (20, 80, 80), (20, 100, 100), (40, 0, 0),
    (40, 0, 40), (40, 20, 40), (40, 20, 60), (40, 40, 0), (40, 40, 20),
    (40, 40, 40), (60, 0, 0), (60, 0, 60), (60, 20, 60), (60, 40, 60),
    (60, 60, 0), (60, 60, 20), (60, 60, 60), (80, 0, 0), (80, 0, 80),
    (80, 20, 80), (80, 60, 80), (80, 80, 0), (80, 80, 20), (80, 80, 80),
    (100, 0, 0), (100, 0, 100), (100, 20, 100), (100, 100, 0),
    (100, 100, 20), (100, 100, 100),
]


def _coverage_band(days_covered: float, lead_time: float) -> int:
    if days_covered < lead_time:
        return 0
    if days_covered < 2.0 * lead_time:
        return 1
    return 2


def _demand_level_band(recent_mean: float, reference_mean: float) -> int:
    if recent_mean < 0.9 * reference_mean:
        return 0
    if recent_mean > 1.1 * reference_mean:
        return 2
    return 1


def _trend_band(trend: float) -> int:
    if trend < -2.0:
        return 0
    if trend > 2.0:
        return 2
    return 1


def _utilisation_band(value: float) -> int:
    if value < 0.5:
        return 0
    if value < 0.85:
        return 1
    return 2


def _discretize_observation(observation) -> list[int]:
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(np.asarray(observation["capacity_utilisation"]).reshape(-1)[0])

    pipeline_total = pipeline.sum(axis=1)
    inventory_position = inventory + pipeline_total
    last3_mean = demand_history[-3:].mean(axis=0)
    last7_mean = demand_history.mean(axis=0)
    trend = last3_mean - last7_mean
    demand_estimate = np.maximum(last3_mean, _EPS)

    bands: list[int] = []
    for product in range(3):
        lead_time = float(_REFERENCE_LEAD_TIMES[product])
        bands.append(_coverage_band(float(inventory_position[product] / demand_estimate[product]), lead_time))
        bands.append(_coverage_band(float(inventory[product] / demand_estimate[product]), lead_time))
        bands.append(_coverage_band(float(pipeline_total[product] / demand_estimate[product]), lead_time))
        bands.append(_demand_level_band(float(last3_mean[product]), float(_REFERENCE_DEMAND_MEANS[product])))
        bands.append(_trend_band(float(trend[product])))

    current_volume = float(np.dot(inventory, _PRODUCT_VOLUMES))
    pipeline_volume = float(np.dot(pipeline_total, _PRODUCT_VOLUMES))
    projected_utilisation = (current_volume + pipeline_volume) / _CAPACITY
    phase = day / _HORIZON
    episode_phase = 0 if phase < (1.0 / 3.0) else (1 if phase < (2.0 / 3.0) else 2)

    bands.append(_utilisation_band(capacity_utilisation))
    bands.append(_utilisation_band(projected_utilisation))
    bands.append(episode_phase)
    return bands


def _flatten_observation(observation) -> np.ndarray:
    bands = _discretize_observation(observation)
    features = np.zeros(FEATURE_DIM, dtype=np.float32)
    for group_index, band in enumerate(bands):
        features[group_index * N_BINS + band] = 1.0
    return features


_WEIGHTS_PATH = Path(__file__).resolve().parent / "policy_weights.npz"
_W = np.load(str(_WEIGHTS_PATH))["W"]


def run_policy(observation):
    """Deterministic (greedy) TD(lambda) inference producing three order quantities."""
    features = _flatten_observation(observation)
    q_values = _W @ features
    action_index = int(np.argmax(q_values))
    return list(ACTION_CATALOGUE[action_index])


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\industrial_inventory_env\__init__.py

"""Official environment package for the IITM RL inventory-control project."""
from .config import (
    PROJECT_VERSION,
    generate_student_config,
    normalize_roll_number,
    public_config_summary,
    validate_student_config,
)
from .environment import IndustrialInventoryEnv


def make_env(
    roll_number: str,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
) -> IndustrialInventoryEnv:
    """Convenience constructor using a roll number."""
    config = generate_student_config(roll_number)
    return IndustrialInventoryEnv(
        config,
        scenario_mode=scenario_mode,
        domain_randomization=domain_randomization,
    )


__all__ = [
    "IndustrialInventoryEnv",
    "PROJECT_VERSION",
    "generate_student_config",
    "make_env",
    "normalize_roll_number",
    "public_config_summary",
    "validate_student_config",
]


In [ ]:
C:\Users\U1124370\Documents\Reinforcement Learning Practice\RL_Project_Student_Package_2026\final_submission\training_pipelines\industrial_inventory_env\config.py

"""Deterministic student-variant generation for the 2026 inventory project.

The generator is intentionally deterministic: the same normalized roll number and
project version always map to the same variant.  The catalogue uses balanced
profiles so that aggregate demand, starting inventory and delay exposure remain
similar across variants while product-level values differ.
"""
from __future__ import annotations

from copy import deepcopy
import hashlib
import json
import re
from typing import Any

PROJECT_VERSION = "IITM-6002W-RL-Inventory-2026-v1"

# All profiles remain within the ranges declared in the problem statement.
_DEMAND_PROFILES = [
    [0.90, 1.00, 1.10],
    [0.90, 1.10, 1.00],
    [1.00, 0.90, 1.10],
    [1.00, 1.10, 0.90],
    [1.10, 0.90, 1.00],
    [1.10, 1.00, 0.90],
    [0.95, 1.00, 1.05],
    [0.95, 1.05, 1.00],
    [1.00, 0.95, 1.05],
    [1.00, 1.05, 0.95],
    [1.05, 0.95, 1.00],
    [1.05, 1.00, 0.95],
    [0.85, 1.00, 1.15],
    [0.85, 1.15, 1.00],
    [1.00, 0.85, 1.15],
    [1.00, 1.15, 0.85],
    [1.15, 0.85, 1.00],
    [1.15, 1.00, 0.85],
]

_INITIAL_INVENTORY_PROFILES = [
    [80, 100, 120],
    [80, 120, 100],
    [100, 80, 120],
    [100, 120, 80],
    [120, 80, 100],
    [120, 100, 80],
    [90, 100, 110],
    [90, 110, 100],
    [100, 90, 110],
    [100, 110, 90],
    [110, 90, 100],
    [110, 100, 90],
]

_DELAY_PROFILES = [
    [0.00, 0.05, 0.10],
    [0.00, 0.10, 0.05],
    [0.05, 0.00, 0.10],
    [0.05, 0.10, 0.00],
    [0.10, 0.00, 0.05],
    [0.10, 0.05, 0.00],
    [0.02, 0.05, 0.08],
    [0.02, 0.08, 0.05],
    [0.05, 0.02, 0.08],
    [0.05, 0.08, 0.02],
    [0.08, 0.02, 0.05],
    [0.08, 0.05, 0.02],
]


def _build_variant_catalogue() -> list[dict[str, Any]]:
    """Create a fixed catalogue of balanced variants.

    The catalogue is generated from constant lists, not from runtime randomness,
    so its contents are stable across machines and Python versions.
    """
    catalogue: list[dict[str, Any]] = []
    count = 36
    for index in range(count):
        demand = _DEMAND_PROFILES[index % len(_DEMAND_PROFILES)]
        inventory = _INITIAL_INVENTORY_PROFILES[(index * 5 + 2) % len(_INITIAL_INVENTORY_PROFILES)]
        delay = _DELAY_PROFILES[(index * 7 + 1) % len(_DELAY_PROFILES)]
        catalogue.append(
            {
                "variant_id": f"V{index + 1:03d}",
                "demand_multiplier_profile": list(demand),
                "initial_inventory_profile": list(inventory),
                "lead_time_delay_profile": list(delay),
            }
        )
    return catalogue


VARIANT_CATALOGUE = _build_variant_catalogue()


def normalize_roll_number(roll_number: str) -> str:
    """Normalize and validate a student roll number."""
    if not isinstance(roll_number, str):
        raise TypeError("roll_number must be a string")

    normalized = re.sub(r"\s+", "", roll_number).upper()
    if not normalized or normalized == "ENTER_YOUR_ROLL_NUMBER":
        raise ValueError("Enter your official roll number before generating the configuration.")
    if not re.fullmatch(r"[A-Z0-9_\-/]{4,40}", normalized):
        raise ValueError(
            "Roll number may contain only letters, digits, underscore, hyphen or slash."
        )
    return normalized


def _fingerprint(payload: dict[str, Any]) -> str:
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:16]


def generate_student_config(roll_number: str) -> dict[str, Any]:
    """Generate the official deterministic configuration for a roll number.

    Parameters
    ----------
    roll_number:
        Official student roll number.

    Returns
    -------
    dict
        A JSON-serialisable configuration dictionary.
    """
    normalized = normalize_roll_number(roll_number)
    digest = hashlib.sha256(
        f"{PROJECT_VERSION}:{normalized}".encode("utf-8")
    ).digest()
    variant_index = int.from_bytes(digest[:8], byteorder="big") % len(VARIANT_CATALOGUE)
    variant = deepcopy(VARIANT_CATALOGUE[variant_index])

    config: dict[str, Any] = {
        "project_version": PROJECT_VERSION,
        "roll_number": normalized,
        **variant,
        "declared_ranges": {
            "demand_multiplier": [0.85, 1.15],
            "initial_inventory": [80, 120],
            "lead_time_delay_probability": [0.00, 0.10],
        },
    }
    config["config_fingerprint"] = _fingerprint(config)
    validate_student_config(config)
    return config


def validate_student_config(config: dict[str, Any]) -> None:
    """Validate structure and declared parameter limits."""
    if not isinstance(config, dict):
        raise TypeError("config must be a dictionary")

    required = {
        "project_version",
        "roll_number",
        "variant_id",
        "demand_multiplier_profile",
        "initial_inventory_profile",
        "lead_time_delay_profile",
    }
    missing = required.difference(config)
    if missing:
        raise ValueError(f"Configuration is missing fields: {sorted(missing)}")

    if config["project_version"] != PROJECT_VERSION:
        raise ValueError(
            f"Configuration project version must be {PROJECT_VERSION!r}."
        )

    normalize_roll_number(config["roll_number"])

    demand = config["demand_multiplier_profile"]
    inventory = config["initial_inventory_profile"]
    delays = config["lead_time_delay_profile"]

    if len(demand) != 3 or not all(0.85 <= float(value) <= 1.15 for value in demand):
        raise ValueError("Demand multiplier profile must contain three values in [0.85, 1.15].")
    if len(inventory) != 3 or not all(
        80 <= int(value) <= 120 and int(value) % 10 == 0 for value in inventory
    ):
        raise ValueError(
            "Initial inventory profile must contain three multiples of 10 in [80, 120]."
        )
    if len(delays) != 3 or not all(0.0 <= float(value) <= 0.10 for value in delays):
        raise ValueError("Delay profile must contain three probabilities in [0.00, 0.10].")


def public_config_summary(config: dict[str, Any]) -> dict[str, Any]:
    """Return the fields students should record in their notebook/report."""
    validate_student_config(config)
    keys = [
        "project_version",
        "roll_number",
        "variant_id",
        "config_fingerprint",
        "demand_multiplier_profile",
        "initial_inventory_profile",
        "lead_time_delay_profile",
    ]
    return {key: deepcopy(config[key]) for key in keys}


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\industrial_inventory_env\environment.py

"""Gymnasium-compatible three-product industrial inventory environment."""
from __future__ import annotations

from copy import deepcopy
from typing import Any, Iterable

import gymnasium as gym
from gymnasium import spaces
import numpy as np

from .config import validate_student_config


class IndustrialInventoryEnv(gym.Env):
    """Multi-product inventory-control environment for the RL course project.

    Internal actions are indices in ``MultiDiscrete([11, 11, 11])``.  Index ``a_i``
    represents an order quantity of ``10 * a_i`` units for product ``i``.

    Parameters
    ----------
    student_config:
        Configuration returned by ``generate_student_config``.
    scenario_mode:
        One of ``"random"``, ``"stationary"``, ``"seasonal"``, ``"trend"``,
        ``"shock"`` or ``"mixed"``.  A sequence such as ``["seasonal", "trend"]``
        is also accepted.
    domain_randomization:
        When True, each episode applies small bounded perturbations around the
        assigned student profile.  All realised values remain within the ranges
        declared in the problem statement.
    """

    metadata = {"render_modes": []}

    NUM_PRODUCTS = 3
    CAPACITY = 1000.0
    PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float64)
    HOLDING_COST_PER_VOLUME = 5.0
    STOCKOUT_COSTS = np.asarray([400.0, 500.0, 300.0], dtype=np.float64)
    FIXED_ORDERING_COSTS = np.asarray([80.0, 200.0, 120.0], dtype=np.float64)
    DISCARDING_COSTS = np.asarray([200.0, 250.0, 150.0], dtype=np.float64)
    REFERENCE_LEAD_TIMES = np.asarray([3, 2, 1], dtype=np.int64)
    REFERENCE_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float64)
    HORIZON = 50
    PIPELINE_DAYS = 4
    DEMAND_HISTORY_DAYS = 7

    _VALID_SCENARIO_COMPONENTS = {"seasonal", "trend", "shock"}

    def __init__(
        self,
        student_config: dict[str, Any],
        scenario_mode: str | Iterable[str] = "random",
        domain_randomization: bool = True,
    ) -> None:
        super().__init__()
        validate_student_config(student_config)
        self.student_config = deepcopy(student_config)
        self.scenario_mode = scenario_mode
        self.domain_randomization = bool(domain_randomization)

        self.action_space = spaces.MultiDiscrete(
            np.asarray([11, 11, 11], dtype=np.int64)
        )
        self.observation_space = spaces.Dict(
            {
                "inventory": spaces.Box(
                    low=0,
                    high=1000,
                    shape=(3,),
                    dtype=np.int32,
                ),
                "arrival_pipeline": spaces.Box(
                    low=0,
                    high=10000,
                    shape=(3, 4),
                    dtype=np.int32,
                ),
                "demand_history": spaces.Box(
                    low=0,
                    high=10000,
                    shape=(7, 3),
                    dtype=np.int32,
                ),
                "day": spaces.Box(
                    low=0,
                    high=self.HORIZON,
                    shape=(1,),
                    dtype=np.int32,
                ),
                "capacity_utilisation": spaces.Box(
                    low=0.0,
                    high=1.0,
                    shape=(1,),
                    dtype=np.float32,
                ),
            }
        )

        self.day = 0
        self.inventory = np.zeros(3, dtype=np.int64)
        self.arrival_pipeline = np.zeros((3, 4), dtype=np.int64)
        self.demand_history = np.zeros((7, 3), dtype=np.int64)
        self.episode_cost = 0.0
        self.episode_parameters: dict[str, Any] = {}
        self._demand_sequence = np.zeros((self.HORIZON, 3), dtype=np.int64)
        self._delay_flags = np.zeros((self.HORIZON, 3), dtype=bool)

    @staticmethod
    def quantities_to_action_indices(quantities: Iterable[int]) -> np.ndarray:
        """Convert actual order quantities to internal action indices."""
        array = np.asarray(list(quantities))
        if array.shape != (3,):
            raise ValueError("Order quantities must contain exactly three values.")
        if not np.issubdtype(array.dtype, np.number):
            raise TypeError("Order quantities must be numeric.")
        if not np.all(np.isfinite(array)):
            raise ValueError("Order quantities must be finite.")
        if not np.all(array == np.round(array)):
            raise ValueError("Order quantities must be integers.")
        array = array.astype(np.int64)
        if not np.all((array >= 0) & (array <= 100) & (array % 10 == 0)):
            raise ValueError("Each order quantity must be one of 0, 10, ..., 100.")
        return array // 10

    @staticmethod
    def action_indices_to_quantities(
        action: Iterable[int],
    ) -> np.ndarray:
        """Convert internal action indices to order quantities."""

        array = np.asarray(list(action))

        if array.shape != (3,):
            raise ValueError(
                "Action must contain exactly three values."
            )

        if not np.issubdtype(array.dtype, np.number):
            raise TypeError("Action indices must be numeric.")

        if not np.all(np.isfinite(array)):
            raise ValueError("Action indices must be finite.")

        if not np.all(array == np.round(array)):
            raise ValueError("Action indices must be integers.")

        array = array.astype(np.int64)

        if not np.all((array >= 0) & (array <= 10)):
            raise ValueError(
                "Action must contain three integer indices in [0, 10]."
            )

        return array * 10

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict[str, Any] | None = None,
    ) -> tuple[dict[str, np.ndarray], dict[str, Any]]:
        super().reset(seed=seed)
        options = options or {}

        self.day = 0
        self.episode_cost = 0.0
        self.arrival_pipeline = np.zeros((3, 4), dtype=np.int64)
        self.demand_history = np.zeros((7, 3), dtype=np.int64)

        self.episode_parameters = self._sample_episode_parameters(options)
        self.inventory = np.asarray(
            self.episode_parameters["initial_inventory"], dtype=np.int64
        ).copy()
        self._prepare_episode_sequences()

        observation = self._get_observation()
        info = self._reset_info(seed)
        return observation, info

    def step(
        self, action: np.ndarray | list[int] | tuple[int, int, int]
    ) -> tuple[dict[str, np.ndarray], float, bool, bool, dict[str, Any]]:
        if self.day >= self.HORIZON:
            raise RuntimeError("Episode is complete. Call reset() before step().")

        action_array = np.asarray(action, dtype=np.int64)
        if not self.action_space.contains(action_array):
            raise ValueError(
                "Environment action must contain three integer indices in [0, 10]. "
                "Use quantities_to_action_indices() when starting from actual quantities."
            )
        order_quantities = action_array * 10

        # 1. Receive orders due today, then advance the future-arrival pipeline.
        scheduled_arrivals = self.arrival_pipeline[:, 0].copy()
        self.arrival_pipeline[:, :-1] = self.arrival_pipeline[:, 1:]
        self.arrival_pipeline[:, -1] = 0

        # 2. Enforce warehouse capacity after arrivals.
        accepted_arrivals, discarded = self._accept_arrivals_with_capacity(
            scheduled_arrivals
        )
        self.inventory += accepted_arrivals

        # 3. Add today's order to its future arrival position.
        realised_delays = self._delay_flags[self.day].astype(np.int64)
        for product in range(self.NUM_PRODUCTS):
            quantity = int(order_quantities[product])
            if quantity == 0:
                continue
            lead_time = int(self.REFERENCE_LEAD_TIMES[product] + realised_delays[product])
            pipeline_index = lead_time - 1
            self.arrival_pipeline[product, pipeline_index] += quantity

        # 4-5. Generate and serve demand.
        demand = self._demand_sequence[self.day].copy()
        fulfilled = np.minimum(self.inventory, demand)
        unfulfilled = demand - fulfilled
        self.inventory -= fulfilled

        # 6. Update history and state summaries.
        self.demand_history[:-1] = self.demand_history[1:]
        self.demand_history[-1] = demand

        # 7. Calculate official costs and reward.
        holding_cost = float(
            np.dot(self.inventory, self.PRODUCT_VOLUMES)
            * self.HOLDING_COST_PER_VOLUME
        )
        stockout_cost = float(np.dot(unfulfilled, self.STOCKOUT_COSTS))
        ordering_cost = float(
            np.dot(order_quantities > 0, self.FIXED_ORDERING_COSTS)
        )
        discard_cost = float(np.dot(discarded, self.DISCARDING_COSTS))
        daily_cost = holding_cost + stockout_cost + ordering_cost + discard_cost
        reward = -daily_cost / 100.0
        self.episode_cost += daily_cost

        current_day = self.day
        self.day += 1
        terminated = False
        truncated = self.day >= self.HORIZON

        observation = self._get_observation()
        info = {
            "day": current_day,
            "demand": demand.astype(np.int32),
            "fulfilled_demand": fulfilled.astype(np.int32),
            "unfulfilled_demand": unfulfilled.astype(np.int32),
            "scheduled_arrivals": scheduled_arrivals.astype(np.int32),
            "accepted_arrivals": accepted_arrivals.astype(np.int32),
            "discarded_units": discarded.astype(np.int32),
            "order_quantities": order_quantities.astype(np.int32),
            "realised_one_day_delay": realised_delays.astype(np.int32),
            "costs": {
                "holding": holding_cost,
                "stockout": stockout_cost,
                "ordering": ordering_cost,
                "discarding": discard_cost,
                "daily_total": daily_cost,
                "episode_total": self.episode_cost,
            },
        }
        return observation, float(reward), terminated, truncated, info

    def _sample_episode_parameters(self, options: dict[str, Any]) -> dict[str, Any]:
        base_demand = np.asarray(
            self.student_config["demand_multiplier_profile"], dtype=np.float64
        )
        base_inventory = np.asarray(
            self.student_config["initial_inventory_profile"], dtype=np.int64
        )
        base_delay = np.asarray(
            self.student_config["lead_time_delay_profile"], dtype=np.float64
        )

        if self.domain_randomization:
            demand_offsets = self.np_random.choice(
                np.asarray([-0.05, 0.0, 0.05]), size=3
            )
            demand_multipliers = np.clip(
                base_demand + demand_offsets, 0.85, 1.15
            )

            inventory_offsets = self.np_random.choice(
                np.asarray([-10, 0, 10]), size=3
            )
            initial_inventory = np.clip(
                base_inventory + inventory_offsets, 80, 120
            ).astype(np.int64)

            delay_offsets = self.np_random.choice(
                np.asarray([-0.02, 0.0, 0.02]), size=3
            )
            delay_probabilities = np.clip(
                base_delay + delay_offsets, 0.0, 0.10
            )
        else:
            demand_multipliers = base_demand.copy()
            initial_inventory = base_inventory.copy()
            delay_probabilities = base_delay.copy()

        scenario_components = self._resolve_scenario_components(
            options.get("scenario_mode", self.scenario_mode)
        )

        return {
            "variant_id": self.student_config["variant_id"],
            "demand_multipliers": demand_multipliers.tolist(),
            "initial_inventory": initial_inventory.tolist(),
            "delay_probabilities": delay_probabilities.tolist(),
            "scenario_components": scenario_components,
        }

    def _resolve_scenario_components(
        self, mode: str | Iterable[str]
    ) -> list[str]:
        if isinstance(mode, str):
            normalized = mode.strip().lower()
            if normalized == "stationary":
                return []
            if normalized in self._VALID_SCENARIO_COMPONENTS:
                return [normalized]
            if normalized == "mixed":
                count = 2
                return sorted(
                    self.np_random.choice(
                        sorted(self._VALID_SCENARIO_COMPONENTS),
                        size=count,
                        replace=False,
                    ).tolist()
                )
            if normalized == "random":
                count = int(self.np_random.choice([0, 1, 2], p=[0.25, 0.50, 0.25]))
                if count == 0:
                    return []
                return sorted(
                    self.np_random.choice(
                        sorted(self._VALID_SCENARIO_COMPONENTS),
                        size=count,
                        replace=False,
                    ).tolist()
                )
            raise ValueError(
                "scenario_mode must be random, stationary, seasonal, trend, shock or mixed."
            )

        components = sorted({str(item).strip().lower() for item in mode})
        invalid = set(components).difference(self._VALID_SCENARIO_COMPONENTS)
        if invalid:
            raise ValueError(f"Unsupported scenario components: {sorted(invalid)}")
        return components

    def _prepare_episode_sequences(self) -> None:
        multipliers = np.asarray(
            self.episode_parameters["demand_multipliers"], dtype=np.float64
        )
        expected = np.tile(
            self.REFERENCE_DEMAND_MEANS * multipliers,
            (self.HORIZON, 1),
        )
        days = np.arange(self.HORIZON, dtype=np.float64)
        components = set(self.episode_parameters["scenario_components"])
        scenario_details: dict[str, Any] = {}

        if "seasonal" in components:
            amplitudes = self.np_random.uniform(0.05, 0.12, size=3)
            phases = self.np_random.uniform(0.0, 2.0 * np.pi, size=3)
            seasonal_factor = 1.0 + amplitudes[None, :] * np.sin(
                2.0 * np.pi * days[:, None] / 7.0 + phases[None, :]
            )
            expected *= seasonal_factor
            scenario_details["seasonal_amplitudes"] = amplitudes.tolist()

        if "trend" in components:
            direction = int(self.np_random.choice([-1, 1]))
            magnitude = float(self.np_random.uniform(0.10, 0.22))
            centered = np.linspace(-0.5, 0.5, self.HORIZON)
            trend_factor = 1.0 + direction * magnitude * centered
            expected *= trend_factor[:, None]
            scenario_details["trend_direction"] = "up" if direction > 0 else "down"
            scenario_details["trend_total_change"] = magnitude

        if "shock" in components:
            product = int(self.np_random.integers(0, self.NUM_PRODUCTS))
            start = int(self.np_random.integers(8, 36))
            duration = int(self.np_random.integers(3, 8))
            factor = float(self.np_random.uniform(1.20, 1.45))
            end = min(start + duration, self.HORIZON)
            expected[start:end, product] *= factor
            scenario_details["shock_product"] = product + 1
            scenario_details["shock_start_day"] = start
            scenario_details["shock_duration"] = end - start
            scenario_details["shock_factor"] = factor

        expected = np.maximum(expected, 0.01)
        self._demand_sequence = self.np_random.poisson(expected).astype(np.int64)
        delay_probabilities = np.asarray(
            self.episode_parameters["delay_probabilities"], dtype=np.float64
        )
        self._delay_flags = (
            self.np_random.random((self.HORIZON, self.NUM_PRODUCTS))
            < delay_probabilities[None, :]
        )
        self.episode_parameters["scenario_details"] = scenario_details

    def _accept_arrivals_with_capacity(
        self, arrivals: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        arrivals = np.asarray(arrivals, dtype=np.int64)
        current_volume = float(np.dot(self.inventory, self.PRODUCT_VOLUMES))
        available_volume = max(self.CAPACITY - current_volume, 0.0)
        arrival_volume = float(np.dot(arrivals, self.PRODUCT_VOLUMES))

        if arrival_volume <= available_volume + 1e-9:
            return arrivals.copy(), np.zeros(3, dtype=np.int64)
        if available_volume <= 0.0:
            return np.zeros(3, dtype=np.int64), arrivals.copy()

        fraction = available_volume / arrival_volume
        raw_acceptance = arrivals.astype(np.float64) * fraction
        accepted = np.floor(raw_acceptance).astype(np.int64)
        used_volume = float(np.dot(accepted, self.PRODUCT_VOLUMES))
        remaining_volume = max(available_volume - used_volume, 0.0)

        fractional_parts = raw_acceptance - accepted
        priority = sorted(
            range(self.NUM_PRODUCTS),
            key=lambda index: (-fractional_parts[index], index),
        )
        made_progress = True
        while made_progress:
            made_progress = False
            for product in priority:
                if accepted[product] >= arrivals[product]:
                    continue
                volume = float(self.PRODUCT_VOLUMES[product])
                if volume <= remaining_volume + 1e-9:
                    accepted[product] += 1
                    remaining_volume -= volume
                    made_progress = True

        discarded = arrivals - accepted
        return accepted, discarded

    def _get_observation(self) -> dict[str, np.ndarray]:
        utilisation = float(
            np.dot(self.inventory, self.PRODUCT_VOLUMES) / self.CAPACITY
        )
        return {
            "inventory": self.inventory.astype(np.int32).copy(),
            "arrival_pipeline": self.arrival_pipeline.astype(np.int32).copy(),
            "demand_history": self.demand_history.astype(np.int32).copy(),
            "day": np.asarray([self.day], dtype=np.int32),
            "capacity_utilisation": np.asarray([utilisation], dtype=np.float32),
        }

    def _reset_info(self, seed: int | None) -> dict[str, Any]:
        return {
            "seed": seed,
            "roll_number": self.student_config["roll_number"],
            "variant_id": self.student_config["variant_id"],
            "config_fingerprint": self.student_config.get("config_fingerprint"),
            "episode_parameters": deepcopy(self.episode_parameters),
        }


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\__init__.py

"""Shared network/buffer/schedule/checkpoint/env-factory building blocks."""



In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\action_catalogue.py

"""Reduced legal action catalogue for TD(lambda)/Tabular SARSA (Phase 5, section 8.2).

The full 1331-action joint space is intractable for a coarse-coded linear
model with per-action weights, so we curate 40-100 "operationally sensible"
quantity combinations from {0,20,40,60,80,100} per product instead of the
full 6**3=216 cross product.
"""
from __future__ import annotations

from itertools import combinations

QUANTITY_LEVELS = (0, 20, 40, 60, 80, 100)


def build_action_catalogue() -> list[tuple[int, int, int]]:
    """Programmatically curate 40-100 legal joint-quantity combinations."""
    actions: set[tuple[int, int, int]] = set()

    # Uniform orders across all three products.
    for level in QUANTITY_LEVELS:
        actions.add((level, level, level))

    # One product elevated from a zero baseline ("order only what's short").
    for position in range(3):
        for level in QUANTITY_LEVELS[1:]:
            action = [0, 0, 0]
            action[position] = level
            actions.add(tuple(action))

    # One product held back below a high uniform base ("skip the
    # well-stocked product while replenishing the rest").
    high_bases = (40, 60, 80, 100)
    low_values = (0, 20)
    for base in high_bases:
        for position in range(3):
            for low in low_values:
                action = [base, base, base]
                action[position] = low
                actions.add(tuple(action))

    # Two products elevated together, the third at a low baseline ("two
    # products are trending up together").
    elevated_values = (40, 60, 80)
    baselines = (0, 20)
    for pair in combinations(range(3), 2):
        for level in elevated_values:
            for baseline in baselines:
                action = [baseline, baseline, baseline]
                for position in pair:
                    action[position] = level
                actions.add(tuple(action))

    # Explicit staggered examples from the problem statement.
    actions.update(
        {
            (0, 0, 0),
            (20, 20, 20),
            (40, 40, 40),
            (60, 40, 60),
            (40, 20, 60),
            (80, 60, 80),
            (100, 100, 100),
        }
    )

    return sorted(actions)


ACTION_CATALOGUE: list[tuple[int, int, int]] = build_action_catalogue()
CATALOGUE_SIZE = len(ACTION_CATALOGUE)


def catalogue_action_to_env_indices(catalogue_index: int) -> list[int]:
    """Convert a catalogue index to environment action indices in [0, 10]."""
    quantities = ACTION_CATALOGUE[catalogue_index]
    return [quantity // 10 for quantity in quantities]


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\checkpoint.py

"""Checkpoint save/load helpers shared by the Phase 4+ custom PyTorch algorithms."""
from __future__ import annotations

from pathlib import Path
from typing import Any

import torch


def save_checkpoint(
    path: str | Path,
    *,
    model_state: dict,
    obs_dim: int,
    n_actions: int,
    hidden_sizes: Any,
    extra: dict[str, Any] | None = None,
) -> None:
    payload = {
        "model_state": model_state,
        "obs_dim": int(obs_dim),
        "n_actions": int(n_actions),
        "hidden_sizes": list(hidden_sizes),
    }
    payload.update(extra or {})
    torch.save(payload, str(path))


def load_checkpoint(path: str | Path) -> dict[str, Any]:
    return torch.load(str(path), map_location="cpu", weights_only=False)


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\discretizer.py

"""Sparse coarse-coded feature discretizer for TD(lambda)/Tabular SARSA (Phase 5, section 8.1).

Turns the observation into "bands" (low/medium/high, encoded 0/1/2) instead
of raw continuous values, then one-hot-encodes each band-group and
concatenates them into one sparse binary feature vector -- a simple, fast
form of coarse coding with a small, fixed number of active features per
state (no full state table required).

Per-product bands (5 x 3 products = 15): inventory position coverage,
on-hand coverage, pipeline coverage, demand level, demand trend.
Global bands (3): capacity utilisation, projected capacity utilisation,
episode phase.
"""
from __future__ import annotations

import numpy as np

from industrial_inventory_env.environment import IndustrialInventoryEnv

_REFERENCE_DEMAND_MEANS = IndustrialInventoryEnv.REFERENCE_DEMAND_MEANS.astype(np.float32)
_REFERENCE_LEAD_TIMES = IndustrialInventoryEnv.REFERENCE_LEAD_TIMES.astype(np.float32)
_PRODUCT_VOLUMES = IndustrialInventoryEnv.PRODUCT_VOLUMES.astype(np.float32)
_CAPACITY = float(IndustrialInventoryEnv.CAPACITY)
_HORIZON = float(IndustrialInventoryEnv.HORIZON)
_NUM_PRODUCTS = IndustrialInventoryEnv.NUM_PRODUCTS
_EPS = 1e-6

N_BINS = 3  # low / medium / high, per band-group

PER_PRODUCT_BAND_NAMES = (
    "inventory_position_coverage",
    "on_hand_coverage",
    "pipeline_coverage",
    "demand_level",
    "demand_trend",
)
GLOBAL_BAND_NAMES = ("capacity_utilisation", "projected_capacity_utilisation", "episode_phase")
N_BAND_GROUPS = len(PER_PRODUCT_BAND_NAMES) * _NUM_PRODUCTS + len(GLOBAL_BAND_NAMES)  # 18
FEATURE_DIM = N_BAND_GROUPS * N_BINS  # 54


def _coverage_band(days_covered: float, lead_time: float) -> int:
    if days_covered < lead_time:
        return 0
    if days_covered < 2.0 * lead_time:
        return 1
    return 2


def _demand_level_band(recent_mean: float, reference_mean: float) -> int:
    if recent_mean < 0.9 * reference_mean:
        return 0
    if recent_mean > 1.1 * reference_mean:
        return 2
    return 1


def _trend_band(trend: float) -> int:
    if trend < -2.0:
        return 0
    if trend > 2.0:
        return 2
    return 1


def _utilisation_band(value: float) -> int:
    if value < 0.5:
        return 0
    if value < 0.85:
        return 1
    return 2


def discretize_observation(observation) -> tuple[int, ...]:
    """Return one band index (0/1/2) per band-group, in a fixed order."""
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(np.asarray(observation["capacity_utilisation"]).reshape(-1)[0])

    pipeline_total = pipeline.sum(axis=1)
    inventory_position = inventory + pipeline_total
    last3_mean = demand_history[-3:].mean(axis=0)
    last7_mean = demand_history.mean(axis=0)
    trend = last3_mean - last7_mean
    demand_estimate = np.maximum(last3_mean, _EPS)

    bands: list[int] = []
    for product in range(_NUM_PRODUCTS):
        lead_time = float(_REFERENCE_LEAD_TIMES[product])
        bands.append(_coverage_band(float(inventory_position[product] / demand_estimate[product]), lead_time))
        bands.append(_coverage_band(float(inventory[product] / demand_estimate[product]), lead_time))
        bands.append(_coverage_band(float(pipeline_total[product] / demand_estimate[product]), lead_time))
        bands.append(_demand_level_band(float(last3_mean[product]), float(_REFERENCE_DEMAND_MEANS[product])))
        bands.append(_trend_band(float(trend[product])))

    current_volume = float(np.dot(inventory, _PRODUCT_VOLUMES))
    pipeline_volume = float(np.dot(pipeline_total, _PRODUCT_VOLUMES))
    projected_utilisation = (current_volume + pipeline_volume) / _CAPACITY
    phase = day / _HORIZON
    episode_phase = 0 if phase < (1.0 / 3.0) else (1 if phase < (2.0 / 3.0) else 2)

    bands.append(_utilisation_band(capacity_utilisation))
    bands.append(_utilisation_band(projected_utilisation))
    bands.append(episode_phase)

    return tuple(bands)


def encode_features(bands: tuple[int, ...]) -> np.ndarray:
    """One-hot encode each band and concatenate into a sparse (FEATURE_DIM,) vector."""
    features = np.zeros(FEATURE_DIM, dtype=np.float32)
    for group_index, band in enumerate(bands):
        features[group_index * N_BINS + band] = 1.0
    return features


def flatten_observation_bands(observation) -> np.ndarray:
    """Discretize + one-hot encode in one call."""
    return encode_features(discretize_observation(observation))


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\env_factory.py

"""Env factory for Phase 4+ custom algorithms: joint action space (1331) +
Representation B features (raw + engineered, 76-dim) per doc section 7.1
("Input: raw plus engineered features").
"""
from __future__ import annotations

from typing import Any

import gymnasium as gym

from industrial_inventory_env import IndustrialInventoryEnv
from training_utils.env_factory import load_assigned_config
from training_utils.reward_shaping import ShapedReward

from src.environment.wrappers import JointActionWrapper
from src.features.engineered import EngineeredObsWrapper


def make_training_env(
    config: dict[str, Any] | None = None,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
    shaping: bool = True,
    shaping_kwargs: dict[str, Any] | None = None,
) -> gym.Env:
    if config is None:
        config = load_assigned_config()
    env = IndustrialInventoryEnv(
        student_config=config, scenario_mode=scenario_mode, domain_randomization=domain_randomization
    )
    if shaping:
        env = ShapedReward(env, **(shaping_kwargs or {}))
    env = JointActionWrapper(env)
    env = EngineeredObsWrapper(env)
    return env


def make_eval_env(
    config: dict[str, Any] | None = None,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
) -> gym.Env:
    """Never shaped -- evaluation always uses the official reward."""
    if config is None:
        config = load_assigned_config()
    env = IndustrialInventoryEnv(
        student_config=config, scenario_mode=scenario_mode, domain_randomization=domain_randomization
    )
    env = JointActionWrapper(env)
    env = EngineeredObsWrapper(env)
    return env


def make_raw_training_env(
    config: dict[str, Any] | None = None,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
    shaping: bool = True,
    shaping_kwargs: dict[str, Any] | None = None,
) -> gym.Env:
    """Base env (Dict obs, MultiDiscrete action) + optional reward shaping --
    for algorithms (TD(lambda), Tabular SARSA) that discretize observations
    and decode a reduced action catalogue themselves (Phase 5, section 8)."""
    if config is None:
        config = load_assigned_config()
    env = IndustrialInventoryEnv(
        student_config=config, scenario_mode=scenario_mode, domain_randomization=domain_randomization
    )
    if shaping:
        env = ShapedReward(env, **(shaping_kwargs or {}))
    return env


def make_raw_eval_env(
    config: dict[str, Any] | None = None,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
) -> gym.Env:
    """Never shaped -- evaluation always uses the official reward."""
    if config is None:
        config = load_assigned_config()
    return IndustrialInventoryEnv(
        student_config=config, scenario_mode=scenario_mode, domain_randomization=domain_randomization
    )



In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\networks.py

"""Joint-action Q-network (Phase 4, section 7.1)."""
from __future__ import annotations

from typing import Sequence

import torch
import torch.nn as nn

DEFAULT_HIDDEN_SIZES: tuple[int, ...] = (256, 256, 128)


class QNetwork(nn.Module):
    """MLP mapping a flat observation to Q-values over the joint action space."""

    def __init__(
        self,
        obs_dim: int,
        n_actions: int,
        hidden_sizes: Sequence[int] = DEFAULT_HIDDEN_SIZES,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        last = obs_dim
        for size in hidden_sizes:
            layers.append(nn.Linear(last, size))
            layers.append(nn.ReLU())
            last = size
        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(last, n_actions)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.head(self.trunk(obs))


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\replay_buffer.py

"""Replay buffer storing full SARSA transitions (Phase 4, section 7.3).

Storing the actual next action taken by the behavior policy (not re-derived
at sample time) is what preserves the SARSA target when using a replay
buffer instead of pure online updates.
"""
from __future__ import annotations

import numpy as np


class SarsaReplayBuffer:
    def __init__(self, capacity: int, obs_dim: int) -> None:
        self.capacity = int(capacity)
        self.obs_dim = int(obs_dim)
        self._obs = np.zeros((self.capacity, self.obs_dim), dtype=np.float32)
        self._actions = np.zeros(self.capacity, dtype=np.int64)
        self._rewards = np.zeros(self.capacity, dtype=np.float32)
        self._next_obs = np.zeros((self.capacity, self.obs_dim), dtype=np.float32)
        self._next_actions = np.zeros(self.capacity, dtype=np.int64)
        self._dones = np.zeros(self.capacity, dtype=np.float32)
        self._size = 0
        self._cursor = 0

    def __len__(self) -> int:
        return self._size

    def add(
        self,
        obs: np.ndarray,
        action: int,
        reward: float,
        next_obs: np.ndarray,
        next_action: int,
        done: bool,
    ) -> None:
        idx = self._cursor
        self._obs[idx] = obs
        self._actions[idx] = action
        self._rewards[idx] = reward
        self._next_obs[idx] = next_obs
        self._next_actions[idx] = next_action
        self._dones[idx] = float(done)
        self._cursor = (self._cursor + 1) % self.capacity
        self._size = min(self._size + 1, self.capacity)

    def sample(self, batch_size: int, rng: np.random.Generator) -> dict[str, np.ndarray]:
        indices = rng.integers(0, self._size, size=batch_size)
        return {
            "obs": self._obs[indices],
            "actions": self._actions[indices],
            "rewards": self._rewards[indices],
            "next_obs": self._next_obs[indices],
            "next_actions": self._next_actions[indices],
            "dones": self._dones[indices],
        }


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\common\schedules.py

"""Parameter schedules (e.g. epsilon-greedy exploration decay)."""
from __future__ import annotations

from typing import Callable


def linear_schedule(start: float, end: float, decay_steps: int) -> Callable[[int], float]:
    """Return step -> value, linearly interpolated from `start` to `end`."""
    decay_steps = max(1, int(decay_steps))

    def _schedule(step: int) -> float:
        frac = min(1.0, max(0.0, step / decay_steps))
        return start + frac * (end - start)

    return _schedule


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\__init__.py

"""Neural-network algorithms shared across Phase 4+ (Neural SARSA, TD(lambda))."""


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\neural_sarsa.py

"""Neural Network SARSA (Phase 4, section 7).

The defining difference from DQN: the TD target evaluates Q at the action
the behavior (epsilon-greedy) policy ACTUALLY takes next, never the greedy
argmax action. A replay buffer is used for sample efficiency; each stored
transition keeps the true next action taken at collection time so the SARSA
target is preserved even though updates are sampled out of order.
"""
from __future__ import annotations

import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import numpy as np
import torch
import torch.nn.functional as F

from src.environment.action_codec import JOINT_ACTION_SIZE

from .common.checkpoint import save_checkpoint
from .common.networks import DEFAULT_HIDDEN_SIZES, QNetwork
from .common.replay_buffer import SarsaReplayBuffer
from .common.schedules import linear_schedule


@dataclass
class NeuralSarsaConfig:
    gamma: float = 0.99
    learning_rate: float = 2e-4
    batch_size: int = 128
    buffer_size: int = 50_000
    learning_starts: int = 5_000
    train_frequency: int = 1
    gradient_steps: int = 1
    target_update_interval: int = 2_000
    epsilon_start: float = 1.0
    epsilon_end: float = 0.03
    epsilon_decay_transitions: int = 300_000
    max_grad_norm: float = 10.0
    hidden_sizes: tuple[int, ...] = DEFAULT_HIDDEN_SIZES
    n_actions: int = JOINT_ACTION_SIZE
    seed: int = 20260825
    total_transitions: int = 250_000
    eval_every: int = 25_000
    eval_seeds: tuple[int, ...] = field(default_factory=lambda: tuple(range(900, 910)))


def _epsilon_greedy_action(
    q_values: np.ndarray, epsilon: float, rng: np.random.Generator, n_actions: int
) -> int:
    if rng.random() < epsilon:
        return int(rng.integers(0, n_actions))
    return int(np.argmax(q_values))


def _evaluate_greedy(q_net: QNetwork, eval_env_fn: Callable[[], Any], seeds) -> tuple[float, float]:
    env = eval_env_fn()
    costs, services = [], []
    for seed in seeds:
        obs, _ = env.reset(seed=int(seed))
        done = False
        ep_cost = 0.0
        ep_demand = ep_fulfilled = 0
        while not done:
            with torch.no_grad():
                q_values = q_net(torch.from_numpy(np.asarray(obs, dtype=np.float32)).unsqueeze(0))
            action = int(torch.argmax(q_values, dim=-1).item())
            obs, _reward, terminated, truncated, info = env.step(action)
            ep_cost += float(info["costs"]["daily_total"])
            ep_demand += int(np.sum(info["demand"]))
            ep_fulfilled += int(np.sum(info["fulfilled_demand"]))
            done = bool(terminated or truncated)
        costs.append(ep_cost)
        services.append(ep_fulfilled / max(ep_demand, 1))
    return float(np.mean(costs)), float(np.mean(services))


def train_neural_sarsa(
    env_fn: Callable[[], Any],
    eval_env_fn: Callable[[], Any],
    save_dir: str | Path,
    cfg: NeuralSarsaConfig = NeuralSarsaConfig(),
    log_writer: Callable[[dict], None] | None = None,
    verbose: bool = True,
) -> dict[str, Any]:
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(cfg.seed)
    torch.manual_seed(cfg.seed)

    env = env_fn()
    obs_dim = int(np.asarray(env.observation_space.shape).prod())

    q_net = QNetwork(obs_dim, cfg.n_actions, cfg.hidden_sizes)
    q_target = QNetwork(obs_dim, cfg.n_actions, cfg.hidden_sizes)
    q_target.load_state_dict(q_net.state_dict())
    q_target.eval()
    optimizer = torch.optim.Adam(q_net.parameters(), lr=cfg.learning_rate)

    buffer = SarsaReplayBuffer(cfg.buffer_size, obs_dim)
    epsilon_schedule = linear_schedule(cfg.epsilon_start, cfg.epsilon_end, cfg.epsilon_decay_transitions)

    def _select_action(obs: np.ndarray, step: int) -> int:
        epsilon = epsilon_schedule(step)
        with torch.no_grad():
            q_values = q_net(torch.from_numpy(obs).unsqueeze(0)).numpy()[0]
        return _epsilon_greedy_action(q_values, epsilon, rng, cfg.n_actions)

    history: list[dict[str, Any]] = []
    best_eval_cost = float("inf")
    n_gradient_updates = 0
    last_loss = float("nan")

    obs, _ = env.reset(seed=cfg.seed)
    obs = np.asarray(obs, dtype=np.float32)
    action = _select_action(obs, 0)

    heartbeat_every = max(1, cfg.total_transitions // 50)
    started_at = time.perf_counter()

    for step in range(1, cfg.total_transitions + 1):
        next_obs, reward, terminated, truncated, _info = env.step(action)
        next_obs = np.asarray(next_obs, dtype=np.float32)
        done = bool(terminated or truncated)
        # The actual action the behavior policy takes next -- this is what
        # makes the update SARSA rather than Q-learning/DQN.
        next_action = _select_action(next_obs, step)

        buffer.add(obs, action, reward, next_obs, next_action, done)

        if done:
            obs, _ = env.reset()
            obs = np.asarray(obs, dtype=np.float32)
            action = _select_action(obs, step)
        else:
            obs = next_obs
            action = next_action

        if (
            step >= cfg.learning_starts
            and step % cfg.train_frequency == 0
            and len(buffer) >= cfg.batch_size
        ):
            for _ in range(cfg.gradient_steps):
                batch = buffer.sample(cfg.batch_size, rng)
                obs_t = torch.from_numpy(batch["obs"])
                actions_t = torch.from_numpy(batch["actions"]).long()
                rewards_t = torch.from_numpy(batch["rewards"])
                next_obs_t = torch.from_numpy(batch["next_obs"])
                next_actions_t = torch.from_numpy(batch["next_actions"]).long()
                dones_t = torch.from_numpy(batch["dones"])

                with torch.no_grad():
                    next_q_all = q_target(next_obs_t)
                    next_q = next_q_all.gather(1, next_actions_t.unsqueeze(1)).squeeze(1)
                    target = rewards_t + cfg.gamma * (1.0 - dones_t) * next_q

                current_q_all = q_net(obs_t)
                current_q = current_q_all.gather(1, actions_t.unsqueeze(1)).squeeze(1)

                loss = F.smooth_l1_loss(current_q, target)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(q_net.parameters(), cfg.max_grad_norm)
                optimizer.step()
                last_loss = float(loss.item())
                n_gradient_updates += 1

                if n_gradient_updates % cfg.target_update_interval == 0:
                    q_target.load_state_dict(q_net.state_dict())

        if verbose and step % heartbeat_every == 0:
            elapsed = time.perf_counter() - started_at
            print(
                f"  ...heartbeat step {step:>8}/{cfg.total_transitions} "
                f"({100 * step / cfg.total_transitions:5.1f}%) "
                f"elapsed={elapsed:6.1f}s fps={step / elapsed:6.1f}",
                flush=True,
            )

        if step % cfg.eval_every == 0 or step == cfg.total_transitions:
            eval_mean_cost, eval_mean_service = _evaluate_greedy(q_net, eval_env_fn, cfg.eval_seeds)
            record = {
                "step": step,
                "loss": last_loss,
                "epsilon": epsilon_schedule(step),
                "eval_mean_cost": eval_mean_cost,
                "eval_mean_service": eval_mean_service,
            }
            if eval_mean_cost < best_eval_cost:
                best_eval_cost = eval_mean_cost
                save_checkpoint(
                    save_dir / "policy_state.pt",
                    model_state=q_net.state_dict(),
                    obs_dim=obs_dim,
                    n_actions=cfg.n_actions,
                    hidden_sizes=cfg.hidden_sizes,
                    extra={"step": step, "eval_mean_cost": eval_mean_cost},
                )
                record["saved_best"] = True
            history.append(record)
            if log_writer is not None:
                log_writer(record)
            if verbose:
                print(
                    f"[step {step:>8}] loss={last_loss:.4f} eps={record['epsilon']:.3f} "
                    f"eval_cost={eval_mean_cost:,.1f} eval_service={eval_mean_service:.3f}"
                    + (" *new best*" if record.get("saved_best") else ""),
                    flush=True,
                )

    save_checkpoint(
        save_dir / "policy_state_final.pt",
        model_state=q_net.state_dict(),
        obs_dim=obs_dim,
        n_actions=cfg.n_actions,
        hidden_sizes=cfg.hidden_sizes,
        extra={"step": cfg.total_transitions},
    )
    return {"history": history, "best_eval_cost": best_eval_cost}


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\algorithms\td_lambda.py

"""True Online SARSA(lambda) with linear function approximation and a reduced
action catalogue (Phase 5, section 8).

Traces reset at episode boundaries (doc 8.3), so training is organized by
full episodes rather than a flat transition counter, unlike Neural SARSA.
Uses the "true online" update (van Seijen et al. 2016 / Sutton & Barto 2nd
ed. section 12.7), which corrects the double-counting that plain
accumulating-trace SARSA(lambda) would introduce under linear function
approximation.
"""
from __future__ import annotations

import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import numpy as np

from .common.action_catalogue import ACTION_CATALOGUE, CATALOGUE_SIZE, catalogue_action_to_env_indices
from .common.discretizer import FEATURE_DIM, flatten_observation_bands


@dataclass
class TDLambdaConfig:
    gamma: float = 0.99
    lambda_: float = 0.85
    alpha: float = 0.01
    epsilon_start: float = 0.4
    epsilon_end: float = 0.02
    epsilon_decay_episodes: int = 10_000
    seed: int = 20260825
    total_episodes: int = 2_000
    eval_every: int = 200
    eval_seeds: tuple[int, ...] = field(default_factory=lambda: tuple(range(900, 910)))


class LinearSarsaLambda:
    """Q(s, a) = W[a] . phi(s), one weight row per catalogue action."""

    def __init__(self, n_actions: int = CATALOGUE_SIZE, n_features: int = FEATURE_DIM) -> None:
        self.n_actions = n_actions
        self.n_features = n_features
        self.W = np.zeros((n_actions, n_features), dtype=np.float64)

    def q_values(self, features: np.ndarray) -> np.ndarray:
        return self.W @ features

    def save(self, path: str | Path) -> None:
        np.savez(str(path), W=self.W)

    @classmethod
    def load(cls, path: str | Path) -> "LinearSarsaLambda":
        data = np.load(str(path))
        W = data["W"]
        model = cls(n_actions=W.shape[0], n_features=W.shape[1])
        model.W = W
        return model


def _epsilon_greedy_action(q_values: np.ndarray, epsilon: float, rng: np.random.Generator) -> int:
    if rng.random() < epsilon:
        return int(rng.integers(0, len(q_values)))
    return int(np.argmax(q_values))


def _epsilon_schedule(episode: int, cfg: TDLambdaConfig) -> float:
    frac = min(1.0, episode / max(1, cfg.epsilon_decay_episodes))
    return cfg.epsilon_start + frac * (cfg.epsilon_end - cfg.epsilon_start)


def _evaluate_greedy(
    model: LinearSarsaLambda, eval_env_fn: Callable[[], Any], seeds
) -> tuple[float, float]:
    env = eval_env_fn()
    costs, services = [], []
    for seed in seeds:
        obs, _ = env.reset(seed=int(seed))
        done = False
        ep_cost = 0.0
        ep_demand = ep_fulfilled = 0
        while not done:
            action_index = int(np.argmax(model.q_values(flatten_observation_bands(obs))))
            obs, _reward, terminated, truncated, info = env.step(
                catalogue_action_to_env_indices(action_index)
            )
            ep_cost += float(info["costs"]["daily_total"])
            ep_demand += int(np.sum(info["demand"]))
            ep_fulfilled += int(np.sum(info["fulfilled_demand"]))
            done = bool(terminated or truncated)
        costs.append(ep_cost)
        services.append(ep_fulfilled / max(ep_demand, 1))
    return float(np.mean(costs)), float(np.mean(services))


def train_td_lambda(
    env_fn: Callable[[], Any],
    eval_env_fn: Callable[[], Any],
    save_dir: str | Path,
    cfg: TDLambdaConfig = TDLambdaConfig(),
    verbose: bool = True,
) -> dict[str, Any]:
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(cfg.seed)

    model = LinearSarsaLambda(CATALOGUE_SIZE, FEATURE_DIM)
    env = env_fn()

    history: list[dict[str, Any]] = []
    best_eval_cost = float("inf")
    started_at = time.perf_counter()

    for episode in range(1, cfg.total_episodes + 1):
        epsilon = _epsilon_schedule(episode, cfg)
        z = np.zeros_like(model.W)

        obs, _ = env.reset(seed=cfg.seed + episode)
        features = flatten_observation_bands(obs)
        action = _epsilon_greedy_action(model.q_values(features), epsilon, rng)
        q_old = 0.0
        done = False

        while not done:
            next_obs, reward, terminated, truncated, _info = env.step(
                catalogue_action_to_env_indices(action)
            )
            done = bool(terminated or truncated)
            next_features = flatten_observation_bands(next_obs)
            next_action = _epsilon_greedy_action(model.q_values(next_features), epsilon, rng)

            q_sa = float(model.W[action] @ features)
            q_next = 0.0 if done else float(model.W[next_action] @ next_features)
            delta = reward + cfg.gamma * q_next - q_sa

            z *= cfg.gamma * cfg.lambda_
            trace_dot = float(z[action] @ features)
            z[action] += (1.0 - cfg.alpha * cfg.gamma * cfg.lambda_ * trace_dot) * features

            model.W += cfg.alpha * (delta + q_sa - q_old) * z
            model.W[action] -= cfg.alpha * (q_sa - q_old) * features

            q_old = q_next
            features = next_features
            action = next_action

        if episode % cfg.eval_every == 0 or episode == cfg.total_episodes:
            eval_mean_cost, eval_mean_service = _evaluate_greedy(model, eval_env_fn, cfg.eval_seeds)
            record = {
                "episode": episode,
                "epsilon": epsilon,
                "eval_mean_cost": eval_mean_cost,
                "eval_mean_service": eval_mean_service,
            }
            if eval_mean_cost < best_eval_cost:
                best_eval_cost = eval_mean_cost
                model.save(save_dir / "policy_weights.npz")
                record["saved_best"] = True
            history.append(record)
            if verbose:
                elapsed = time.perf_counter() - started_at
                print(
                    f"[episode {episode:>6}/{cfg.total_episodes}] eps={epsilon:.3f} "
                    f"eval_cost={eval_mean_cost:,.1f} eval_service={eval_mean_service:.3f} "
                    f"elapsed={elapsed:.1f}s" + (" *new best*" if record.get("saved_best") else ""),
                    flush=True,
                )

    model.save(save_dir / "policy_weights_final.npz")
    return {"history": history, "best_eval_cost": best_eval_cost}


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\environment\__init__.py

"""Action-space codec and wrappers shared by every algorithm (Phase 1, section 4.1)."""
from .action_codec import (
    JOINT_ACTION_SIZE,
    indices_to_quantities,
    joint_index_to_multidiscrete,
    joint_index_to_quantities,
    multidiscrete_to_joint_index,
    quantities_to_indices,
    quantities_to_joint_index,
)
from .wrappers import JointActionWrapper

__all__ = [
    "JOINT_ACTION_SIZE",
    "JointActionWrapper",
    "indices_to_quantities",
    "joint_index_to_multidiscrete",
    "joint_index_to_quantities",
    "multidiscrete_to_joint_index",
    "quantities_to_indices",
    "quantities_to_joint_index",
]


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\environment\action_codec.py

"""Single source of truth for action conversion (Phase 1, section 4.1).

The official environment action is ``MultiDiscrete([11, 11, 11])``: index
``a_i`` means an order quantity of ``10 * a_i`` units for product ``i``. DQN
family algorithms instead act over one joint ``Discrete(1331)`` action space
using the base-11 encoding below. Every algorithm and policy submission must
go through these functions -- never re-derive the encoding locally.

Encoding:      joint_index = a1 * 121 + a2 * 11 + a3
Inverse:       a1 = joint_index // 121
               remainder = joint_index % 121
               a2 = remainder // 11
               a3 = remainder % 11
"""
from __future__ import annotations

from typing import Iterable

NUM_PRODUCTS = 3
INDICES_PER_PRODUCT = 11
QUANTITY_STEP = 10
MAX_QUANTITY = 100
JOINT_ACTION_SIZE = INDICES_PER_PRODUCT**NUM_PRODUCTS  # 1331


def _as_int_list(values: Iterable, expected_len: int, label: str) -> list[int]:
    values = list(values)
    if len(values) != expected_len:
        raise ValueError(f"Expected {expected_len} {label}, got {len(values)}.")
    return [int(v) for v in values]


def indices_to_quantities(action_indices: Iterable[int]) -> list[int]:
    """Convert [0..10, 0..10, 0..10] indices to quantities [0, 10, ..., 100]."""
    indices = _as_int_list(action_indices, NUM_PRODUCTS, "action indices")
    for index in indices:
        if not 0 <= index <= INDICES_PER_PRODUCT - 1:
            raise ValueError(f"Action index {index} out of range [0, 10].")
    return [index * QUANTITY_STEP for index in indices]


def quantities_to_indices(order_quantities: Iterable[int]) -> list[int]:
    """Convert official order quantities [0, 10, ..., 100] to action indices."""
    quantities = _as_int_list(order_quantities, NUM_PRODUCTS, "order quantities")
    indices = []
    for quantity in quantities:
        if quantity < 0 or quantity > MAX_QUANTITY or quantity % QUANTITY_STEP != 0:
            raise ValueError(f"Order quantity {quantity} is not one of 0, 10, ..., 100.")
        indices.append(quantity // QUANTITY_STEP)
    return indices


def joint_index_to_multidiscrete(joint_index: int) -> list[int]:
    """Convert one of 1331 joint actions to three indices [a1, a2, a3]."""
    joint_index = int(joint_index)
    if not 0 <= joint_index < JOINT_ACTION_SIZE:
        raise ValueError(f"joint_index must be in [0, {JOINT_ACTION_SIZE}).")
    a1 = joint_index // 121
    remainder = joint_index % 121
    a2 = remainder // 11
    a3 = remainder % 11
    return [a1, a2, a3]


def multidiscrete_to_joint_index(action_indices: Iterable[int]) -> int:
    """Convert three indices [a1, a2, a3] to one joint action index."""
    a1, a2, a3 = _as_int_list(action_indices, NUM_PRODUCTS, "action indices")
    for index in (a1, a2, a3):
        if not 0 <= index <= INDICES_PER_PRODUCT - 1:
            raise ValueError(f"Action index {index} out of range [0, 10].")
    return a1 * 121 + a2 * 11 + a3


def joint_index_to_quantities(joint_index: int) -> list[int]:
    """Compose joint_index_to_multidiscrete + indices_to_quantities."""
    return indices_to_quantities(joint_index_to_multidiscrete(joint_index))


def quantities_to_joint_index(order_quantities: Iterable[int]) -> int:
    """Compose quantities_to_indices + multidiscrete_to_joint_index."""
    return multidiscrete_to_joint_index(quantities_to_indices(order_quantities))


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\environment\wrappers.py

"""Action-space wrapper built on the canonical codec (Phase 1, section 4.1 / 2).

Does not touch observations, rewards, or transition dynamics -- only recodes
the action the agent sees, per the "do not modify the official environment
logic" constraint.
"""
from __future__ import annotations

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from .action_codec import (
    JOINT_ACTION_SIZE,
    joint_index_to_multidiscrete,
    multidiscrete_to_joint_index,
)


class JointActionWrapper(gym.ActionWrapper):
    """Expose ``Discrete(1331)`` to the agent; the env still receives ``MultiDiscrete([11,11,11])``.

    Required for DQN-family algorithms, whose standard implementations only
    support a single ``Discrete`` action space.
    """

    def __init__(self, env: gym.Env) -> None:
        super().__init__(env)
        self.action_space = spaces.Discrete(JOINT_ACTION_SIZE)

    def action(self, action):
        indices = joint_index_to_multidiscrete(int(action))
        return np.asarray(indices, dtype=np.int64)

    def reverse_action(self, action):
        return multidiscrete_to_joint_index(np.asarray(action, dtype=np.int64).tolist())


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\features\__init__.py

"""Observation feature representations shared by every algorithm (Phase 1, section 4.2)."""
from .engineered import (
    ENGINEERED_FEATURE_DIM,
    REPRESENTATION_B_DIM,
    EngineeredObsWrapper,
    flatten_observation_engineered,
    flatten_observation_representation_b,
)
from .normalizer import DEFAULT_SCALES, NormalizationScales
from .observation import RAW_FEATURE_DIM, RawObsWrapper, flatten_observation_raw

__all__ = [
    "DEFAULT_SCALES",
    "ENGINEERED_FEATURE_DIM",
    "RAW_FEATURE_DIM",
    "REPRESENTATION_B_DIM",
    "EngineeredObsWrapper",
    "NormalizationScales",
    "RawObsWrapper",
    "flatten_observation_engineered",
    "flatten_observation_raw",
    "flatten_observation_representation_b",
]


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\features\engineered.py

"""Representation B: raw + engineered features (Phase 1, section 4.2; formulas
per the iteration-3 "RL Final Run" doc, Phase 3).

Built only from the documented observation dict -- never from `info` or
future demand. Physical constants (capacity, lead times, volumes) are read
directly from `IndustrialInventoryEnv` so they can never drift out of sync
with the official environment.

Per-product engineered features (10 x 3 products = 30 values, product-major):
    inventory_position, recent_3day_mean_demand, recent_7day_mean_demand,
    demand_trend, demand_std, pipeline_within_lead_time,
    pipeline_after_lead_time, lead_time_demand_estimate,
    inventory_position_gap, estimated_days_of_supply

Global engineered features (8 values):
    current_inventory_volume_ratio, capacity_headroom_ratio,
    pipeline_volume_ratio, projected_capacity_utilisation,
    remaining_episode_fraction, episode_phase_early, episode_phase_middle,
    episode_phase_late

Early-episode demand means (`recent_3day_mean_demand`, `recent_7day_mean_demand`,
`demand_std`) ignore the zero-padded rows of `demand_history` that exist before
enough real days have elapsed -- do not let that padding bias the mean toward
zero.
"""
from __future__ import annotations

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from industrial_inventory_env.environment import IndustrialInventoryEnv

from .observation import RAW_FEATURE_DIM, flatten_observation_raw

_PRODUCT_VOLUMES = IndustrialInventoryEnv.PRODUCT_VOLUMES.astype(np.float32)
_CAPACITY = float(IndustrialInventoryEnv.CAPACITY)
_HORIZON = float(IndustrialInventoryEnv.HORIZON)
_REFERENCE_LEAD_TIMES = IndustrialInventoryEnv.REFERENCE_LEAD_TIMES.astype(np.float32)
_NUM_PRODUCTS = IndustrialInventoryEnv.NUM_PRODUCTS
_DEMAND_HISTORY_DAYS = IndustrialInventoryEnv.DEMAND_HISTORY_DAYS
_EPS = 1e-6

PER_PRODUCT_FEATURE_NAMES = (
    "inventory_position",
    "recent_3day_mean_demand",
    "recent_7day_mean_demand",
    "demand_trend",
    "demand_std",
    "pipeline_within_lead_time",
    "pipeline_after_lead_time",
    "lead_time_demand_estimate",
    "inventory_position_gap",
    "estimated_days_of_supply",
)
PER_PRODUCT_FEATURE_DIM = len(PER_PRODUCT_FEATURE_NAMES) * _NUM_PRODUCTS  # 30

GLOBAL_FEATURE_NAMES = (
    "current_inventory_volume_ratio",
    "capacity_headroom_ratio",
    "pipeline_volume_ratio",
    "projected_capacity_utilisation",
    "remaining_episode_fraction",
    "episode_phase_early",
    "episode_phase_middle",
    "episode_phase_late",
)
GLOBAL_FEATURE_DIM = len(GLOBAL_FEATURE_NAMES)  # 8

ENGINEERED_FEATURE_DIM = PER_PRODUCT_FEATURE_DIM + GLOBAL_FEATURE_DIM  # 38
REPRESENTATION_B_DIM = RAW_FEATURE_DIM + ENGINEERED_FEATURE_DIM  # 76


def compute_per_product_features(observation) -> np.ndarray:
    """Return the 10 per-product engineered features, flattened product-major (30,)."""
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = int(np.asarray(observation["day"]).reshape(-1)[0])

    inventory_position = inventory + pipeline.sum(axis=1)

    # `demand_history` is a rolling window (oldest first, most recent last) that
    # is zero-padded at the start of an episode; only the last `valid_days` rows
    # are real observations.
    valid_days = min(day, _DEMAND_HISTORY_DAYS)
    if valid_days == 0:
        last3_mean = np.zeros(_NUM_PRODUCTS, dtype=np.float32)
        last7_mean = np.zeros(_NUM_PRODUCTS, dtype=np.float32)
        demand_std = np.zeros(_NUM_PRODUCTS, dtype=np.float32)
    else:
        valid_history = demand_history[-valid_days:]
        valid3 = valid_history[-min(valid_days, 3):]
        last3_mean = valid3.mean(axis=0)
        last7_mean = valid_history.mean(axis=0)
        demand_std = valid_history.std(axis=0)
    demand_trend = last3_mean - last7_mean

    within_lead_time = np.zeros(_NUM_PRODUCTS, dtype=np.float32)
    after_lead_time = np.zeros(_NUM_PRODUCTS, dtype=np.float32)
    for product in range(_NUM_PRODUCTS):
        lead_time = int(_REFERENCE_LEAD_TIMES[product])
        within_lead_time[product] = pipeline[product, :lead_time].sum()
        after_lead_time[product] = pipeline[product, lead_time:].sum()

    lead_time_demand_estimate = last7_mean * _REFERENCE_LEAD_TIMES
    inventory_position_gap = inventory_position - lead_time_demand_estimate
    estimated_days_of_supply = inventory_position / np.maximum(last7_mean, 1.0)

    per_product = np.stack(
        [
            inventory_position,
            last3_mean,
            last7_mean,
            demand_trend,
            demand_std,
            within_lead_time,
            after_lead_time,
            lead_time_demand_estimate,
            inventory_position_gap,
            estimated_days_of_supply,
        ],
        axis=1,
    )  # shape (num_products, 10)
    return per_product.reshape(-1).astype(np.float32)


def compute_global_features(observation) -> np.ndarray:
    """Return the 8 global engineered features (8,)."""
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])

    current_volume = float(np.dot(inventory, _PRODUCT_VOLUMES))
    pipeline_volume = float(np.dot(pipeline.sum(axis=1), _PRODUCT_VOLUMES))

    current_volume_ratio = current_volume / _CAPACITY
    headroom_ratio = max(_CAPACITY - current_volume, 0.0) / _CAPACITY
    pipeline_volume_ratio = pipeline_volume / _CAPACITY
    projected_utilisation = (current_volume + pipeline_volume) / _CAPACITY
    remaining_fraction = max(_HORIZON - day, 0.0) / _HORIZON

    phase = day / _HORIZON
    early = 1.0 if phase < (1.0 / 3.0) else 0.0
    middle = 1.0 if (1.0 / 3.0) <= phase < (2.0 / 3.0) else 0.0
    late = 1.0 if phase >= (2.0 / 3.0) else 0.0

    return np.asarray(
        [
            current_volume_ratio,
            headroom_ratio,
            pipeline_volume_ratio,
            projected_utilisation,
            remaining_fraction,
            early,
            middle,
            late,
        ],
        dtype=np.float32,
    )


def flatten_observation_engineered(observation) -> np.ndarray:
    """Return the 38 engineered features: 30 per-product + 8 global."""
    return np.concatenate(
        [compute_per_product_features(observation), compute_global_features(observation)]
    ).astype(np.float32)


def flatten_observation_representation_b(observation) -> np.ndarray:
    """Representation B: raw (38) + engineered (38) = 76 features."""
    return np.concatenate(
        [flatten_observation_raw(observation), flatten_observation_engineered(observation)]
    ).astype(np.float32)


class EngineeredObsWrapper(gym.ObservationWrapper):
    """Observation wrapper producing the fixed-size Representation B vector."""

    def __init__(self, env: gym.Env) -> None:
        super().__init__(env)
        self.observation_space = spaces.Box(
            low=-1000.0, high=1000.0, shape=(REPRESENTATION_B_DIM,), dtype=np.float32
        )

    def observation(self, observation):
        return flatten_observation_representation_b(observation)


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\features\normalizer.py

"""Fixed, environment-aware normalization scales (Phase 1, section 4.2).

Scales are constants, never statistics fit on leaderboard episodes, so
training-time and inference-time normalization are always identical.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path


@dataclass(frozen=True)
class NormalizationScales:
    """Divisors applied to raw observation fields (Representation A)."""

    inventory_scale: float = 200.0
    pipeline_scale: float = 100.0
    demand_history_scale: float = 100.0
    day_scale: float = 49.0

    def to_json(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2), encoding="utf-8")

    @classmethod
    def from_json(cls, path: str | Path) -> "NormalizationScales":
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        return cls(**data)


DEFAULT_SCALES = NormalizationScales()


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\features\observation.py

"""Representation A: raw, normalized observation features (Phase 1, section 4.2).

Fixed flattening order (38 raw values total):
    inventory (3), arrival_pipeline (12), demand_history (21), day (1),
    capacity_utilisation (1)
"""
from __future__ import annotations

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from .normalizer import DEFAULT_SCALES, NormalizationScales

RAW_FEATURE_DIM = 38


def flatten_observation_raw(
    observation, scales: NormalizationScales = DEFAULT_SCALES
) -> np.ndarray:
    """Flatten and normalize the official observation dict into a (38,) float32 vector."""
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(
        np.asarray(observation["capacity_utilisation"]).reshape(-1)[0]
    )

    features = np.concatenate(
        [
            inventory / scales.inventory_scale,
            pipeline.reshape(-1) / scales.pipeline_scale,
            demand_history.reshape(-1) / scales.demand_history_scale,
            np.asarray([day / scales.day_scale], dtype=np.float32),
            np.asarray([capacity_utilisation], dtype=np.float32),
        ]
    )
    return features.astype(np.float32, copy=False)


class RawObsWrapper(gym.ObservationWrapper):
    """Observation wrapper producing the fixed-size Representation A vector."""

    def __init__(self, env: gym.Env, scales: NormalizationScales = DEFAULT_SCALES) -> None:
        super().__init__(env)
        self._scales = scales
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(RAW_FEATURE_DIM,), dtype=np.float32
        )

    def observation(self, observation):
        return flatten_observation_raw(observation, self._scales)


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\src\__init__.py

"""Iteration-2 shared foundation: action codec, feature representations, reproducibility utils."""


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_scripts\__init__.py



In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_scripts\common.py

"""Shared helpers for training scripts."""
from __future__ import annotations

from pathlib import Path
from typing import Callable

import gymnasium as gym
import numpy as np
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from training_utils import (
    load_assigned_config,
    make_eval_env,
    make_training_env,
)

PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODELS_DIR = PROJECT_ROOT / "models"
LOGS_DIR = PROJECT_ROOT / "logs"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)


def build_training_vec_env(
    n_envs: int = 4,
    *,
    flatten_action: bool = False,
    shaping: bool = True,
    seed: int = 20260727,
    shaping_kwargs: dict | None = None,
) -> DummyVecEnv:
    """Vectorized training env, in-process (DummyVecEnv) — CPU friendly on Windows."""
    config = load_assigned_config()

    def _factory(rank: int) -> Callable[[], gym.Env]:
        def _make() -> gym.Env:
            env = make_training_env(
                config,
                scenario_mode="random",
                domain_randomization=True,
                flatten_action=flatten_action,
                shaping=shaping,
                shaping_kwargs=shaping_kwargs,
            )
            env = Monitor(env)
            env.reset(seed=seed + 1000 * rank)
            return env

        return _make

    return DummyVecEnv([_factory(i) for i in range(n_envs)])


def build_eval_vec_env(
    n_envs: int = 1,
    *,
    flatten_action: bool = False,
    seed: int = 12345,
    scenario_mode: str = "random",
) -> DummyVecEnv:
    config = load_assigned_config()

    def _factory(rank: int) -> Callable[[], gym.Env]:
        def _make() -> gym.Env:
            env = make_eval_env(
                config,
                scenario_mode=scenario_mode,
                domain_randomization=True,
                flatten_action=flatten_action,
            )
            env = Monitor(env)
            env.reset(seed=seed + rank)
            return env

        return _make

    return DummyVecEnv([_factory(i) for i in range(n_envs)])


__all__ = [
    "EvalCallback",
    "LOGS_DIR",
    "MODELS_DIR",
    "build_eval_vec_env",
    "build_training_vec_env",
]


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_scripts\train_a2c.py

"""Train an A2C agent on the shaped training env.

Deliberately kept methodologically distinct from PPO: no clipping, no multi-epoch
policy reuse, short n_steps rollouts, GAE lambda=1.0 (undiscounted advantage).
"""
from __future__ import annotations

import argparse

from stable_baselines3 import A2C

from training_scripts.common import (
    EvalCallback,
    MODELS_DIR,
    build_eval_vec_env,
    build_training_vec_env,
)


def main() -> int:
    parser = argparse.ArgumentParser(description="Train A2C for inventory control")
    parser.add_argument("--timesteps", type=int, default=200_000)
    parser.add_argument("--n-envs", type=int, default=4)
    parser.add_argument("--seed", type=int, default=20260727)
    parser.add_argument("--eval-freq", type=int, default=10_000)
    parser.add_argument("--n-eval-episodes", type=int, default=10)
    args = parser.parse_args()

    save_dir = MODELS_DIR / "a2c"
    save_dir.mkdir(parents=True, exist_ok=True)

    train_env = build_training_vec_env(
        n_envs=args.n_envs,
        flatten_action=False,
        shaping=True,
        seed=args.seed,
        shaping_kwargs={"anneal_steps": args.timesteps},
    )
    eval_env = build_eval_vec_env(n_envs=1, flatten_action=False, seed=99)

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(save_dir),
        log_path=str(save_dir / "eval_logs"),
        eval_freq=max(args.eval_freq // args.n_envs, 1),
        n_eval_episodes=args.n_eval_episodes,
        deterministic=True,
        verbose=1,
    )

    model = A2C(
        "MlpPolicy",
        train_env,
        learning_rate=7e-4,
        n_steps=32,
        gamma=0.98,
        gae_lambda=1.0,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        use_rms_prop=True,
        policy_kwargs={"net_arch": [128, 128]},
        seed=args.seed,
        verbose=1,
        device="cpu",
    )
    model.learn(total_timesteps=args.timesteps, callback=eval_callback)
    model.save(str(save_dir / "final_model"))
    print(f"A2C training done. Best model at: {save_dir / 'best_model.zip'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_scripts\train_dqn.py

"""Train a DQN agent on the shaped training env with a flattened Discrete(1331)
action space.
"""
from __future__ import annotations

import argparse

from stable_baselines3 import DQN

from training_scripts.common import (
    EvalCallback,
    MODELS_DIR,
    build_eval_vec_env,
    build_training_vec_env,
)


def main() -> int:
    parser = argparse.ArgumentParser(description="Train DQN for inventory control")
    parser.add_argument("--timesteps", type=int, default=150_000)
    parser.add_argument("--seed", type=int, default=20260727)
    parser.add_argument("--eval-freq", type=int, default=10_000)
    parser.add_argument("--n-eval-episodes", type=int, default=10)
    args = parser.parse_args()

    save_dir = MODELS_DIR / "dqn"
    save_dir.mkdir(parents=True, exist_ok=True)

    # DQN is a single-env algorithm.
    train_env = build_training_vec_env(
        n_envs=1,
        flatten_action=True,
        shaping=True,
        seed=args.seed,
        shaping_kwargs={"anneal_steps": args.timesteps},
    )
    eval_env = build_eval_vec_env(n_envs=1, flatten_action=True, seed=99)

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(save_dir),
        log_path=str(save_dir / "eval_logs"),
        eval_freq=args.eval_freq,
        n_eval_episodes=args.n_eval_episodes,
        deterministic=True,
        verbose=1,
    )

    model = DQN(
        "MlpPolicy",
        train_env,
        learning_rate=5e-4,
        buffer_size=200_000,
        learning_starts=5_000,
        batch_size=256,
        tau=1.0,
        gamma=0.98,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=2_000,
        exploration_fraction=0.3,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.02,
        max_grad_norm=10.0,
        policy_kwargs={"net_arch": [256, 256]},
        seed=args.seed,
        verbose=1,
        device="cpu",
    )
    model.learn(total_timesteps=args.timesteps, callback=eval_callback)
    model.save(str(save_dir / "final_model"))
    print(f"DQN training done. Best model at: {save_dir / 'best_model.zip'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_utils\__init__.py

"""Shared training utilities for the IITM RL inventory-control project.

This package intentionally has no side effects on import so that the code inside
each submitted `policy.py` can copy the small preprocessing functions inline
(the evaluator sandbox will not have this package available).
"""
from .obs_wrapper import FEATURE_DIM, FlattenObs, flatten_observation
from .action_wrapper import ACTION_SPACE_SIZE, FlattenAction, joint_index_to_quantities
from .reward_shaping import ShapedReward
from .env_factory import load_assigned_config, make_eval_env, make_training_env

__all__ = [
    "ACTION_SPACE_SIZE",
    "FEATURE_DIM",
    "FlattenAction",
    "FlattenObs",
    "ShapedReward",
    "flatten_observation",
    "joint_index_to_quantities",
    "load_assigned_config",
    "make_eval_env",
    "make_training_env",
]


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_utils\action_wrapper.py

"""Action-flattening wrapper mapping Discrete(1331) <-> MultiDiscrete([11,11,11]).

Enables SB3 DQN (which supports only Discrete action spaces) to operate on the
project's multi-discrete action space. The base-11 encoding is inline-copyable
by each submitted `policy.py`.
"""
from __future__ import annotations

import gymnasium as gym
import numpy as np
from gymnasium import spaces

ACTION_SPACE_SIZE = 11 * 11 * 11  # 1331


def joint_index_to_quantities(index: int) -> list[int]:
    """Decode a joint action index into three order quantities in {0,10,...,100}.

    Kept dependency-free so it can be copied verbatim into each policy file.
    """
    index = int(index)
    a2 = index % 11
    index //= 11
    a1 = index % 11
    a0 = index // 11
    return [a0 * 10, a1 * 10, a2 * 10]


def _quantities_to_joint_index(action_indices: np.ndarray) -> int:
    a0, a1, a2 = (int(x) for x in action_indices)
    return a0 * 121 + a1 * 11 + a2


class FlattenAction(gym.ActionWrapper):
    """Wrap MultiDiscrete([11,11,11]) as Discrete(1331) using base-11 encoding."""

    def __init__(self, env: gym.Env) -> None:
        super().__init__(env)
        self.action_space = spaces.Discrete(ACTION_SPACE_SIZE)

    def action(self, action):
        quantities = joint_index_to_quantities(int(action))
        return np.asarray(
            [q // 10 for q in quantities], dtype=np.int64
        )

    def reverse_action(self, action):
        return _quantities_to_joint_index(np.asarray(action, dtype=np.int64))


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_utils\env_factory.py

"""Env factories used by every training/evaluation script.

`make_training_env` layers optional wrappers in this order:
    IndustrialInventoryEnv -> ShapedReward (optional) -> FlattenObs
      -> FlattenAction (optional, DQN only)

`make_eval_env` never applies reward shaping so evaluation always uses the
official reward per section 11 of the problem statement.
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import gymnasium as gym

from industrial_inventory_env import IndustrialInventoryEnv

from .action_wrapper import FlattenAction
from .obs_wrapper import FlattenObs
from .reward_shaping import ShapedReward

_DEFAULT_CONFIG_PATH = Path(__file__).resolve().parent.parent / "assigned_config.json"


def load_assigned_config(path: str | Path | None = None) -> dict[str, Any]:
    """Load the persisted student configuration written during Phase 0."""
    config_path = Path(path) if path is not None else _DEFAULT_CONFIG_PATH
    with open(config_path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def _base_env(
    config: dict[str, Any],
    scenario_mode: str = "random",
    domain_randomization: bool = True,
) -> gym.Env:
    return IndustrialInventoryEnv(
        student_config=config,
        scenario_mode=scenario_mode,
        domain_randomization=domain_randomization,
    )


def make_training_env(
    config: dict[str, Any] | None = None,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
    flatten_action: bool = False,
    shaping: bool = True,
    shaping_kwargs: dict[str, Any] | None = None,
) -> gym.Env:
    """Build a wrapped environment suitable for training."""
    if config is None:
        config = load_assigned_config()
    env = _base_env(config, scenario_mode, domain_randomization)
    if shaping:
        env = ShapedReward(env, **(shaping_kwargs or {}))
    env = FlattenObs(env)
    if flatten_action:
        env = FlattenAction(env)
    return env


def make_eval_env(
    config: dict[str, Any] | None = None,
    *,
    scenario_mode: str = "random",
    domain_randomization: bool = True,
    flatten_action: bool = False,
) -> gym.Env:
    """Build a wrapped environment suitable for evaluation.

    Reward shaping is never applied — evaluation uses the official reward only.
    """
    if config is None:
        config = load_assigned_config()
    env = _base_env(config, scenario_mode, domain_randomization)
    env = FlattenObs(env)
    if flatten_action:
        env = FlattenAction(env)
    return env


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_utils\obs_wrapper.py

"""Observation-flattening wrapper and standalone helper.

The helper `flatten_observation` is deliberately dependency-free so that each
submitted `policy.py` can copy it verbatim without importing this package.
"""
from __future__ import annotations

import gymnasium as gym
import numpy as np
from gymnasium import spaces

# Product-level constants mirror the ones exposed by the environment.
_PRODUCT_VOLUMES = np.asarray([2.0, 3.0, 1.5], dtype=np.float32)
_CAPACITY = 1000.0
_REF_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float32)

# Feature layout (kept as documentation and to detect drift):
# 0:3   inventory / 100
# 3:15  arrival_pipeline / 100 (row-major)
# 15:18 mean of last 3 days of demand / reference mean
# 18:21 mean of last 7 days of demand / reference mean
# 21:24 std of last 7 days of demand / reference mean
# 24:25 day / 50
# 25:26 capacity_utilisation
# 26:29 inventory position (inventory + pipeline sum) / 200
# 29:32 inventory position in volume / capacity
# 32:35 (inventory position - 3-day-mean-demand) / 100  (safety cushion signal)
FEATURE_DIM = 35


def flatten_observation(observation) -> np.ndarray:
    """Convert the environment observation dict to a float32 feature vector.

    Duplicated in each policy file so that submissions do not depend on this
    package at evaluation time.
    """
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)
    day = float(np.asarray(observation["day"]).reshape(-1)[0])
    capacity_utilisation = float(
        np.asarray(observation["capacity_utilisation"]).reshape(-1)[0]
    )

    last3_mean = demand_history[-3:].mean(axis=0)
    last7_mean = demand_history.mean(axis=0)
    last7_std = demand_history.std(axis=0)

    inv_position_units = inventory + pipeline.sum(axis=1)
    inv_position_volume = inv_position_units * _PRODUCT_VOLUMES

    features = np.concatenate(
        [
            inventory / 100.0,
            pipeline.reshape(-1) / 100.0,
            last3_mean / _REF_DEMAND_MEANS,
            last7_mean / _REF_DEMAND_MEANS,
            last7_std / _REF_DEMAND_MEANS,
            np.asarray([day / 50.0], dtype=np.float32),
            np.asarray([capacity_utilisation], dtype=np.float32),
            inv_position_units / 200.0,
            inv_position_volume / _CAPACITY,
            (inv_position_units - last3_mean) / 100.0,
        ],
        dtype=np.float32,
    )
    return features.astype(np.float32, copy=False)


class FlattenObs(gym.ObservationWrapper):
    """Observation wrapper that produces a fixed-size float32 Box observation."""

    def __init__(self, env: gym.Env) -> None:
        super().__init__(env)
        self.observation_space = spaces.Box(
            low=-10.0,
            high=10.0,
            shape=(FEATURE_DIM,),
            dtype=np.float32,
        )

    def observation(self, observation):
        return flatten_observation(observation)


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\training_utils\reward_shaping.py

"""Reward shaping wrapper for training.

The wrapper strictly adds signals derived from the same `info` dict already
returned by the environment. The baseline reward is preserved.

Coefficients are annealed linearly to zero across the specified number of
environment steps so the final training regime matches the official reward.
"""
from __future__ import annotations

import gymnasium as gym
import numpy as np


class ShapedReward(gym.Wrapper):
    """Add differentiable, monotonically-annealed shaping to the base reward.

    Shaping components (per timestep):
      + alpha * (service_level - 1.0)                # penalise stockouts extra
      - beta  * max(0, capacity_utilisation - 0.85)  # discourage discards
      - gamma * n_orders_placed / 3                  # gently favour batching
    """

    def __init__(
        self,
        env: gym.Env,
        alpha: float = 1.0,
        beta: float = 2.0,
        gamma: float = 0.15,
        anneal_steps: int = 300_000,
    ) -> None:
        super().__init__(env)
        self.alpha_start = float(alpha)
        self.beta_start = float(beta)
        self.gamma_start = float(gamma)
        self.anneal_steps = max(1, int(anneal_steps))
        self._global_step = 0

    def _anneal_factor(self) -> float:
        frac = 1.0 - self._global_step / self.anneal_steps
        return float(max(0.0, frac))

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self._global_step += 1
        anneal = self._anneal_factor()

        # Service level per product; guard against zero demand days.
        demand = np.asarray(info["demand"], dtype=np.float32)
        fulfilled = np.asarray(info["fulfilled_demand"], dtype=np.float32)
        with np.errstate(divide="ignore", invalid="ignore"):
            service = np.where(demand > 0, fulfilled / demand, 1.0)
        service_level = float(service.mean())

        # Post-step capacity_utilisation is available on the returned observation.
        capacity_utilisation = float(
            np.asarray(obs["capacity_utilisation"]).reshape(-1)[0]
        )
        n_orders = float(np.count_nonzero(info["order_quantities"]))

        shaping = (
            self.alpha_start * (service_level - 1.0)
            - self.beta_start * max(0.0, capacity_utilisation - 0.85)
            - self.gamma_start * (n_orders / 3.0)
        ) * anneal

        info = dict(info)
        info["base_reward"] = reward
        info["shaping"] = shaping
        info["anneal_factor"] = anneal
        return obs, reward + shaping, terminated, truncated, info


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\assigned_config.json

{
  "project_version": "IITM-6002W-RL-Inventory-2026-v1",
  "roll_number": "DA25M579",
  "variant_id": "V030",
  "demand_multiplier_profile": [
    1.05,
    1.0,
    0.95
  ],
  "initial_inventory_profile": [
    100,
    120,
    80
  ],
  "lead_time_delay_profile": [
    0.0,
    0.05,
    0.1
  ],
  "declared_ranges": {
    "demand_multiplier": [
      0.85,
      1.15
    ],
    "initial_inventory": [
      80,
      120
    ],
    "lead_time_delay_probability": [
      0.0,
      0.1
    ]
  },
  "config_fingerprint": "5566e180ff842b10"
}

In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\run_neural_sarsa_promote_v2.py

"""Promote the top 2 Neural SARSA v3 screening configs across 3 seeds each
(iteration-3 doc Phase 5 promotion rule / Phase 10 Round 3). Reads
eval_results/phase5_v3/neural_sarsa_screening_results.json (produced by
run_neural_sarsa_screening_v2.py) and re-trains the top-2 configs at a
longer budget, 3 seeds each (6 runs total).
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

from run_neural_sarsa_screening_v2 import CONFIGS, train_one_config

PROJECT_ROOT = Path(__file__).resolve().parent
RESULTS_DIR = PROJECT_ROOT / "eval_results" / "phase5_v3"
MODELS_DIR = PROJECT_ROOT / "models" / "neural_sarsa_v3"

PROMOTE_TRANSITIONS = 500_000
SEEDS = (20260825, 20260826, 20260827)


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--top-n", type=int, default=2)
    parser.add_argument("--total-transitions", type=int, default=PROMOTE_TRANSITIONS)
    args = parser.parse_args()

    screening_path = RESULTS_DIR / "neural_sarsa_screening_results.json"
    screening_results = json.loads(screening_path.read_text(encoding="utf-8"))
    top_configs = screening_results[: args.top_n]  # already sorted lowest-cost-first
    print(f"Promoting top {args.top_n} configs: {[c['config_id'] for c in top_configs]}", flush=True)

    results = []
    for entry in top_configs:
        config_id = entry["config_id"]
        params = {k: entry[k] for k in ("learning_rate", "gamma", "epsilon_end")}
        for seed in SEEDS:
            run_id = f"promoted_{config_id}_seed{seed}"
            save_dir = MODELS_DIR / run_id
            print(f"=== {run_id} {params} ({args.total_transitions} transitions) ===", flush=True)

            # train_one_config always uses SEED=20260825 internally for env/torch
            # seeding via NeuralSarsaConfig.seed -- override per-seed here.
            import run_neural_sarsa_screening_v2 as _mod

            original_seed = _mod.SEED
            _mod.SEED = seed
            try:
                row = train_one_config(config_id, params, args.total_transitions, save_dir)
            finally:
                _mod.SEED = original_seed
            row["seed"] = seed
            row["run_id"] = run_id
            results.append(row)
            print(f"  best_eval_cost={row['best_eval_cost']:.1f}", flush=True)

    with open(RESULTS_DIR / "neural_sarsa_promoted_results.json", "w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2)

    print("\n=== Promoted results (all runs) ===")
    for row in sorted(results, key=lambda r: r["best_eval_cost"]):
        print(row)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\run_neural_sarsa_screening_v2.py

"""Neural SARSA re-screening (iteration-3 "RL Final Run" doc, Phase 5).

Uses the FIXED Representation B features (76-dim, zero-padding + DOS bugs
corrected -- see src/features/engineered.py) and the doc's exact SARSA-A/B/C
screening configs. Does not touch the iteration-2 `models/neural_sarsa/`
checkpoints -- saves to a separate `models/neural_sarsa_v3/` tree.

Promotion rule (doc): mean cost < 140k, service >= 96%, no scenario mean
above 190k -> promote top 2 configs across 3 seeds each (see
run_neural_sarsa_promote_v2.py).
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

from src.algorithms.common.env_factory import make_eval_env, make_training_env
from src.algorithms.neural_sarsa import NeuralSarsaConfig, train_neural_sarsa

PROJECT_ROOT = Path(__file__).resolve().parent
MODELS_DIR = PROJECT_ROOT / "models" / "neural_sarsa_v3"
RESULTS_DIR = PROJECT_ROOT / "eval_results" / "phase5_v3"

SCREEN_TRANSITIONS = 300_000
SEED = 20260825

# Doc's exact SARSA-A/B/C screening configs.
CONFIGS = {
    "sarsa_a": {"learning_rate": 2e-4, "gamma": 0.99, "epsilon_end": 0.03},
    "sarsa_b": {"learning_rate": 1e-4, "gamma": 0.99, "epsilon_end": 0.05},
    "sarsa_c": {"learning_rate": 3e-4, "gamma": 0.98, "epsilon_end": 0.03},
}


def train_one_config(config_id: str, params: dict, total_transitions: int, save_dir: Path) -> dict:
    save_dir.mkdir(parents=True, exist_ok=True)
    cfg = NeuralSarsaConfig(
        gamma=params["gamma"],
        learning_rate=params["learning_rate"],
        epsilon_end=params["epsilon_end"],
        epsilon_decay_transitions=int(0.8 * total_transitions),  # keep decay inside the run
        seed=SEED,
        total_transitions=total_transitions,
    )

    def env_fn():
        return make_training_env(shaping=True, shaping_kwargs={"anneal_steps": total_transitions})

    def eval_env_fn():
        return make_eval_env()

    result = train_neural_sarsa(env_fn, eval_env_fn, save_dir, cfg, verbose=True)
    return {
        "config_id": config_id,
        **params,
        "best_eval_cost": result["best_eval_cost"],
    }


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--total-transitions", type=int, default=SCREEN_TRANSITIONS)
    args = parser.parse_args()

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results = []
    for config_id, params in CONFIGS.items():
        save_dir = MODELS_DIR / config_id
        print(f"=== training {config_id} {params} ({args.total_transitions} transitions) ===", flush=True)
        row = train_one_config(config_id, params, args.total_transitions, save_dir)
        results.append(row)
        print(f"  best_eval_cost={row['best_eval_cost']:.1f}", flush=True)

    results.sort(key=lambda r: r["best_eval_cost"])  # lower cost first
    with open(RESULTS_DIR / "neural_sarsa_screening_results.json", "w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2)

    print("\n=== Ranked screening results (lowest cost first) ===")
    for row in results:
        print(row)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\run_ppo_promote.py

"""Promote the top screening configs to a longer PPO run (Phase 6, doc 9.2:
"Promote the top two, then train to 1,500,000 to 3,000,000 transitions").
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

from run_ppo_screening import MODELS_DIR, RESULTS_DIR, train_one_config


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--timesteps", type=int, default=1_500_000)
    parser.add_argument("--top-n", type=int, default=2)
    args = parser.parse_args()

    with open(RESULTS_DIR / "ppo_screening_results.json", "r", encoding="utf-8") as handle:
        screening_results = json.load(handle)

    top_configs = screening_results[: args.top_n]
    print(f"Promoting top {len(top_configs)} configs to {args.timesteps} timesteps:")
    for row in top_configs:
        print(f"  ent_coef={row['ent_coef']} lr={row['learning_rate']} (screening best={row['best_mean_reward']:.1f})")

    final_results = []
    for row in top_configs:
        ent_coef, lr = row["ent_coef"], row["learning_rate"]
        config_id = f"promoted_ent{ent_coef}_lr{lr}"
        save_dir = MODELS_DIR / config_id
        print(f"\n=== training {config_id} ({args.timesteps} timesteps) ===", flush=True)
        result = train_one_config(ent_coef, lr, args.timesteps, save_dir)
        final_results.append(result)
        print(f"  best_mean_reward={result['best_mean_reward']:.3f}", flush=True)

    final_results.sort(key=lambda r: r["best_mean_reward"], reverse=True)
    with open(RESULTS_DIR / "ppo_promoted_results.json", "w", encoding="utf-8") as handle:
        json.dump(final_results, handle, indent=2)

    print("\n=== Final promoted results (best mean reward first) ===")
    for row in final_results:
        print(row)
    print(f"\nChampion config: {final_results[0]}")


if __name__ == "__main__":
    main()


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\run_ppo_screening.py

"""PPO recovery: screening sweep over entropy coefficient x learning rate
(Phase 6, doc section 9.2). Does NOT touch iteration-1's `train_ppo.py` or
its Phase-0-protected model -- saves to a separate `models/ppo_v2/` tree.

Core settings held constant per doc: n_envs=8, n_steps=512, batch_size=512,
n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, vf_coef=0.5,
max_grad_norm=0.5, normalize_advantage=True. Screens at 500,000 transitions;
promote the top 2 configs afterwards via run_ppo_full_training.py.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback

from training_scripts.common import build_eval_vec_env, build_training_vec_env

PROJECT_ROOT = Path(__file__).resolve().parent
MODELS_DIR = PROJECT_ROOT / "models" / "ppo_v2"
RESULTS_DIR = PROJECT_ROOT / "eval_results" / "phase6"

ENT_COEFS = (0.0, 0.001, 0.003)
LEARNING_RATES = (1e-4, 3e-4)
SCREEN_TIMESTEPS = 500_000
N_ENVS = 8
SEED = 20260825


def _config_id(ent_coef: float, lr: float) -> str:
    return f"ent{ent_coef}_lr{lr}"


def train_one_config(ent_coef: float, lr: float, timesteps: int, save_dir: Path) -> dict:
    save_dir.mkdir(parents=True, exist_ok=True)

    train_env = build_training_vec_env(
        n_envs=N_ENVS,
        flatten_action=False,
        shaping=True,
        seed=SEED,
        shaping_kwargs={"anneal_steps": timesteps},
    )
    eval_env = build_eval_vec_env(n_envs=1, flatten_action=False, seed=99)

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(save_dir),
        log_path=str(save_dir / "eval_logs"),
        eval_freq=max(25_000 // N_ENVS, 1),
        n_eval_episodes=10,
        deterministic=True,
        verbose=0,
    )

    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=lr,
        n_steps=512,
        batch_size=512,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=ent_coef,
        vf_coef=0.5,
        max_grad_norm=0.5,
        normalize_advantage=True,
        policy_kwargs={"net_arch": [128, 128]},
        seed=SEED,
        verbose=0,
        device="cpu",
    )
    model.learn(total_timesteps=timesteps, callback=eval_callback)
    model.save(str(save_dir / "final_model"))

    best_mean_reward = float(eval_callback.best_mean_reward)
    return {"ent_coef": ent_coef, "learning_rate": lr, "best_mean_reward": best_mean_reward}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--timesteps", type=int, default=SCREEN_TIMESTEPS)
    args = parser.parse_args()

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results = []
    for ent_coef in ENT_COEFS:
        for lr in LEARNING_RATES:
            config_id = _config_id(ent_coef, lr)
            save_dir = MODELS_DIR / config_id
            print(f"=== training {config_id} ({args.timesteps} timesteps) ===", flush=True)
            row = train_one_config(ent_coef, lr, args.timesteps, save_dir)
            results.append(row)
            print(f"  best_mean_reward={row['best_mean_reward']:.3f}", flush=True)

    results.sort(key=lambda r: r["best_mean_reward"], reverse=True)  # higher reward = lower cost
    with open(RESULTS_DIR / "ppo_screening_results.json", "w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2)

    print("\n=== Ranked screening results (best mean reward first) ===")
    for row in results:
        print(row)


if __name__ == "__main__":
    main()


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\run_td_lambda_grid_search.py


"""Grid search over TD(lambda)'s starting grid (doc section 8.3):
gamma in [0.98, 0.99], lambda in [0.7, 0.85, 0.95], alpha in [0.005, 0.01, 0.02].
"""
from __future__ import annotations

import json
from pathlib import Path

from src.algorithms.common.env_factory import make_raw_eval_env, make_raw_training_env
from src.algorithms.td_lambda import TDLambdaConfig, train_td_lambda

PROJECT_ROOT = Path(__file__).resolve().parent
RESULTS_DIR = PROJECT_ROOT / "eval_results" / "phase5"

GAMMAS = (0.98, 0.99)
LAMBDAS = (0.7, 0.85, 0.95)
ALPHAS = (0.005, 0.01, 0.02)
TOTAL_EPISODES = 2000
EVAL_EVERY = 500
EPSILON_DECAY_EPISODES = 1600  # 80% of TOTAL_EPISODES


def main() -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results = []
    for gamma in GAMMAS:
        for lambda_ in LAMBDAS:
            for alpha in ALPHAS:
                cfg = TDLambdaConfig(
                    gamma=gamma,
                    lambda_=lambda_,
                    alpha=alpha,
                    total_episodes=TOTAL_EPISODES,
                    eval_every=EVAL_EVERY,
                    epsilon_decay_episodes=EPSILON_DECAY_EPISODES,
                )
                save_dir = RESULTS_DIR / f"g{gamma}_l{lambda_}_a{alpha}"
                result = train_td_lambda(
                    lambda: make_raw_training_env(shaping=True, shaping_kwargs={"anneal_steps": TOTAL_EPISODES * 50}),
                    lambda: make_raw_eval_env(),
                    save_dir,
                    cfg,
                    verbose=False,
                )
                final_record = result["history"][-1]
                row = {
                    "gamma": gamma,
                    "lambda": lambda_,
                    "alpha": alpha,
                    "best_eval_cost": result["best_eval_cost"],
                    "final_eval_cost": final_record["eval_mean_cost"],
                    "final_eval_service": final_record["eval_mean_service"],
                }
                results.append(row)
                print(
                    f"gamma={gamma} lambda={lambda_} alpha={alpha:<6} "
                    f"best_cost={row['best_eval_cost']:>10,.1f} "
                    f"final_cost={row['final_eval_cost']:>10,.1f} "
                    f"final_service={row['final_eval_service']:.3f}"
                )

    results.sort(key=lambda r: r["best_eval_cost"])
    with open(RESULTS_DIR / "grid_search_results.json", "w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2)

    print("\n=== Top 5 configurations by best_eval_cost ===")
    for row in results[:5]:
        print(row)


if __name__ == "__main__":
    main()


In [ ]:
RL_Project_Student_Package_2026\final_submission\training_pipelines\run_td_lambda_three_seeds.py

"""Train the best TD(lambda) config from the grid search across 3 seeds
(doc's "stable across at least 3 seeds" promotion criterion, section 8.4).
"""
from __future__ import annotations

import json
from pathlib import Path

from src.algorithms.common.env_factory import make_raw_eval_env, make_raw_training_env
from src.algorithms.td_lambda import TDLambdaConfig, train_td_lambda

PROJECT_ROOT = Path(__file__).resolve().parent
MODELS_DIR = PROJECT_ROOT / "models" / "td_lambda"
RESULTS_DIR = PROJECT_ROOT / "eval_results" / "phase5"

BEST_GAMMA = 0.98
BEST_LAMBDA = 0.7
BEST_ALPHA = 0.01
TOTAL_EPISODES = 5000
EVAL_EVERY = 250
EPSILON_DECAY_EPISODES = 4000  # 80% of TOTAL_EPISODES
SEEDS = (20260825, 20260826, 20260827)


def main() -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results = []
    for seed in SEEDS:
        cfg = TDLambdaConfig(
            gamma=BEST_GAMMA,
            lambda_=BEST_LAMBDA,
            alpha=BEST_ALPHA,
            total_episodes=TOTAL_EPISODES,
            eval_every=EVAL_EVERY,
            epsilon_decay_episodes=EPSILON_DECAY_EPISODES,
            seed=seed,
        )
        save_dir = MODELS_DIR / f"seed{seed}"
        result = train_td_lambda(
            lambda: make_raw_training_env(shaping=True, shaping_kwargs={"anneal_steps": TOTAL_EPISODES * 50}),
            lambda: make_raw_eval_env(),
            save_dir,
            cfg,
            verbose=False,
        )
        final_record = result["history"][-1]
        row = {
            "seed": seed,
            "best_eval_cost": result["best_eval_cost"],
            "final_eval_cost": final_record["eval_mean_cost"],
            "final_eval_service": final_record["eval_mean_service"],
        }
        results.append(row)
        print(
            f"seed={seed} best_cost={row['best_eval_cost']:>10,.1f} "
            f"final_cost={row['final_eval_cost']:>10,.1f} "
            f"final_service={row['final_eval_service']:.3f}",
            flush=True,
        )

    with open(RESULTS_DIR / "three_seed_results.json", "w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2)

    costs = [r["best_eval_cost"] for r in results]
    services = [r["final_eval_service"] for r in results]
    print(f"\nMean best cost across seeds: {sum(costs)/len(costs):,.1f}")
    print(f"Std best cost across seeds: {(sum((c - sum(costs)/len(costs))**2 for c in costs)/len(costs))**0.5:,.1f}")
    print(f"Mean final service across seeds: {sum(services)/len(services):.3f}")
    best_seed = min(results, key=lambda r: r["best_eval_cost"])["seed"]
    print(f"Best seed: {best_seed}")


if __name__ == "__main__":
    main()


In [ ]:
RL_Project_Student_Package_2026\final_submission\baselines.py

"""Two heuristic baseline policies used as sanity-check floors.

Every RL submission must strictly outperform both baselines on the holdout
evaluation grid; if not, treat it as a bug and investigate.
"""
from __future__ import annotations

import numpy as np

_REF_DEMAND_MEANS = np.asarray([30.0, 25.0, 35.0], dtype=np.float32)
_LEAD_TIMES = np.asarray([3, 2, 1], dtype=np.float32)


def zero_order_policy(observation: dict) -> list[int]:
    """Never order — the worst-case cost floor."""
    return [0, 0, 0]


def order_up_to_policy(observation: dict) -> list[int]:
    """Generalized order-up-to heuristic using the observation's demand history.

    Target inventory position = max(recent demand mean, reference mean) *
    (lead_time + safety_days). Order the shortfall, rounded up to a multiple
    of 10 and clipped to [0, 100].
    """
    inventory = np.asarray(observation["inventory"], dtype=np.float32)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=np.float32)
    demand_history = np.asarray(observation["demand_history"], dtype=np.float32)

    recent = demand_history[-3:].mean(axis=0)
    demand_estimate = np.maximum(recent, _REF_DEMAND_MEANS)

    safety_days = 2.0
    target_position = demand_estimate * (_LEAD_TIMES + safety_days)

    inventory_position = inventory + pipeline.sum(axis=1)
    shortfall = np.maximum(target_position - inventory_position, 0.0)
    order = np.ceil(shortfall / 10.0) * 10.0
    order = np.clip(order, 0.0, 100.0)
    return order.astype(int).tolist()


__all__ = ["order_up_to_policy", "zero_order_policy"]

In [ ]:
RL_Project_Student_Package_2026\final_submission\evaluation.py

"""Local evaluation harness for the IITM RL inventory-control project.

Runs a policy across a grid of (seed x scenario_mode) episodes on the OFFICIAL
(unshaped) environment and returns a tidy DataFrame plus a per-scenario summary.

A "policy" here is any callable `(observation_dict) -> [q1, q2, q3]` returning
quantities in {0, 10, ..., 100}. This is exactly the leaderboard interface, so
the harness can be used against both raw callables and imported `run_policy`
functions from a submission file.

Usage
-----
    from evaluation import evaluate_policy, HOLDOUT_SEEDS, SCENARIO_MODES
    df, summary = evaluate_policy(my_policy, seeds=HOLDOUT_SEEDS, scenario_modes=SCENARIO_MODES)
"""
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Any, Callable, Iterable

import numpy as np
import pandas as pd

from industrial_inventory_env import IndustrialInventoryEnv

PolicyFn = Callable[[dict], Iterable[int]]

# Reserved seeds — never used during training.
HOLDOUT_SEEDS: list[int] = list(range(900, 940))
# Smaller set for quick in-training callbacks.
FAST_EVAL_SEEDS: list[int] = list(range(900, 910))

SCENARIO_MODES: list[str] = [
    "stationary",
    "seasonal",
    "trend",
    "shock",
    "random",
]


def _load_config(config: dict[str, Any] | str | Path | None) -> dict[str, Any]:
    if isinstance(config, dict):
        return config
    if config is None:
        config = Path(__file__).resolve().parent / "assigned_config.json"
    with open(config, "r", encoding="utf-8") as handle:
        return json.load(handle)


def _validate_action(action: Any) -> list[int]:
    if isinstance(action, np.ndarray):
        action = action.tolist()
    if not isinstance(action, (list, tuple)):
        raise TypeError("Policy must return a list, tuple or numpy array.")
    if len(action) != 3:
        raise ValueError("Policy must return exactly three quantities.")
    out: list[int] = []
    for value in action:
        if isinstance(value, (bool, np.bool_)):
            raise TypeError("Order quantities must be integers, not bool.")
        integer = int(value)
        if integer not in range(0, 101, 10):
            raise ValueError(f"Order quantity {integer} is not in 0..100 step 10.")
        out.append(integer)
    return out


def _run_episode(
    env: IndustrialInventoryEnv,
    policy: PolicyFn,
    seed: int,
) -> dict[str, Any]:
    obs, reset_info = env.reset(seed=int(seed))
    total_cost = 0.0
    holding = stockout = ordering = discarding = 0.0
    total_demand = 0
    total_fulfilled = 0
    total_discarded = 0
    sum_capacity_util = 0.0
    step_count = 0
    call_time_sum = 0.0

    done = False
    while not done:
        started = time.perf_counter()
        quantities = _validate_action(policy(obs))
        call_time_sum += time.perf_counter() - started

        action_indices = env.quantities_to_action_indices(quantities)
        obs, reward, terminated, truncated, info = env.step(action_indices)

        total_cost += info["costs"]["daily_total"]
        holding += info["costs"]["holding"]
        stockout += info["costs"]["stockout"]
        ordering += info["costs"]["ordering"]
        discarding += info["costs"]["discarding"]

        total_demand += int(np.sum(info["demand"]))
        total_fulfilled += int(np.sum(info["fulfilled_demand"]))
        total_discarded += int(np.sum(info["discarded_units"]))
        sum_capacity_util += float(obs["capacity_utilisation"][0])
        step_count += 1
        done = bool(terminated or truncated)

    service_level = total_fulfilled / max(total_demand, 1)
    return {
        "seed": int(seed),
        "episode_cost": float(total_cost),
        "holding_cost": float(holding),
        "stockout_cost": float(stockout),
        "ordering_cost": float(ordering),
        "discarding_cost": float(discarding),
        "service_level": float(service_level),
        "total_demand": int(total_demand),
        "total_fulfilled": int(total_fulfilled),
        "total_discarded": int(total_discarded),
        "avg_capacity_utilisation": float(sum_capacity_util / max(step_count, 1)),
        "steps": int(step_count),
        "avg_policy_call_ms": float(1000.0 * call_time_sum / max(step_count, 1)),
        "variant_id": reset_info["variant_id"],
        "config_fingerprint": reset_info.get("config_fingerprint"),
    }


def evaluate_policy(
    policy: PolicyFn,
    *,
    seeds: Iterable[int] = HOLDOUT_SEEDS,
    scenario_modes: Iterable[str] = SCENARIO_MODES,
    config: dict[str, Any] | str | Path | None = None,
    domain_randomization: bool = True,
    progress: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evaluate a policy across the full seed x scenario_mode grid.

    Returns
    -------
    per_episode : pd.DataFrame
        One row per (seed, scenario_mode) combination.
    summary : pd.DataFrame
        Aggregated by scenario_mode: mean/std of total episode cost, service
        level, cost breakdown, and average policy call latency.
    """
    student_config = _load_config(config)
    rows: list[dict[str, Any]] = []
    seed_list = list(seeds)
    for scenario_mode in scenario_modes:
        env = IndustrialInventoryEnv(
            student_config=student_config,
            scenario_mode=scenario_mode,
            domain_randomization=domain_randomization,
        )
        for seed in seed_list:
            row = _run_episode(env, policy, seed)
            row["scenario_mode"] = scenario_mode
            rows.append(row)
            if progress:
                print(f"{scenario_mode} seed={seed} cost={row['episode_cost']:.1f}")
    per_episode = pd.DataFrame(rows)
    summary = (
        per_episode.groupby("scenario_mode")
        .agg(
            mean_cost=("episode_cost", "mean"),
            std_cost=("episode_cost", "std"),
            p50_cost=("episode_cost", "median"),
            p90_cost=("episode_cost", lambda s: float(np.percentile(s, 90))),
            mean_service_level=("service_level", "mean"),
            mean_holding=("holding_cost", "mean"),
            mean_stockout=("stockout_cost", "mean"),
            mean_ordering=("ordering_cost", "mean"),
            mean_discarding=("discarding_cost", "mean"),
            mean_call_ms=("avg_policy_call_ms", "mean"),
        )
        .reset_index()
    )
    return per_episode, summary


def summarise_overall(per_episode: pd.DataFrame) -> dict[str, float]:
    """Return a single-line summary averaged across all evaluated episodes."""
    return {
        "mean_cost": float(per_episode["episode_cost"].mean()),
        "std_cost": float(per_episode["episode_cost"].std()),
        "median_cost": float(per_episode["episode_cost"].median()),
        "mean_service_level": float(per_episode["service_level"].mean()),
        "mean_call_ms": float(per_episode["avg_policy_call_ms"].mean()),
        "n_episodes": int(len(per_episode)),
    }


__all__ = [
    "FAST_EVAL_SEEDS",
    "HOLDOUT_SEEDS",
    "SCENARIO_MODES",
    "evaluate_policy",
    "summarise_overall",
]


In [ ]:
RL_Project_Student_Package_2026\final_submission\policy_validation_tests.py

"""Local validation checks for an inventory-project policy submission.

Usage
-----
python policy_validation_tests_updated.py submission_template.py
python policy_validation_tests_updated.py my_policy.py --max-seconds-per-call 0.25

What this script checks
-----------------------
1. The submitted file is a Python (.py) file and can be imported.
2. It defines exactly one required policy function: run_policy(observation).
3. The function accepts the documented observation dictionary.
4. It returns exactly three integer order quantities.
5. Each quantity is one of 0, 10, 20, ..., 100.
6. It does not modify the observation supplied by the evaluator.
7. It behaves deterministically when reloaded and given the same observation sequence.
8. It does not directly call env.step() or env.reset().
9. Its average inference time is below the configured local limit.

Important
---------
- This script validates only the submission interface.
- Passing these checks does not guarantee leaderboard eligibility or good performance.
- The actual leaderboard evaluator may apply additional security, packaging and runtime checks.
- If the policy requires a trained model file, place that file in the location and format
  specified separately by the course team.
"""
from __future__ import annotations

import argparse
import ast
from copy import deepcopy
import importlib.util
import inspect
from pathlib import Path
import sys
import time
from types import ModuleType
from typing import Any
import uuid

import numpy as np

VALID_QUANTITIES = set(range(0, 101, 10))


def representative_observations() -> list[dict[str, np.ndarray]]:
    """Return representative observations matching the documented interface."""
    return [
        {
            "inventory": np.asarray([100, 100, 100], dtype=np.int32),
            "arrival_pipeline": np.zeros((3, 4), dtype=np.int32),
            "demand_history": np.zeros((7, 3), dtype=np.int32),
            "day": np.asarray([0], dtype=np.int32),
            "capacity_utilisation": np.asarray([0.65], dtype=np.float32),
        },
        {
            "inventory": np.asarray([20, 35, 15], dtype=np.int32),
            "arrival_pipeline": np.asarray(
                [[0, 20, 40, 0], [10, 0, 30, 0], [40, 20, 0, 0]],
                dtype=np.int32,
            ),
            "demand_history": np.asarray(
                [
                    [28, 23, 36],
                    [35, 24, 31],
                    [31, 27, 42],
                    [29, 21, 37],
                    [34, 28, 39],
                    [33, 26, 35],
                    [38, 29, 44],
                ],
                dtype=np.int32,
            ),
            "day": np.asarray([18], dtype=np.int32),
            "capacity_utilisation": np.asarray([0.1675], dtype=np.float32),
        },
        {
            "inventory": np.asarray([180, 150, 120], dtype=np.int32),
            "arrival_pipeline": np.asarray(
                [[60, 20, 0, 0], [50, 0, 0, 0], [80, 0, 0, 0]],
                dtype=np.int32,
            ),
            "demand_history": np.asarray(
                [
                    [30, 25, 34],
                    [29, 24, 37],
                    [31, 26, 33],
                    [30, 25, 36],
                    [28, 27, 35],
                    [33, 24, 32],
                    [30, 25, 35],
                ],
                dtype=np.int32,
            ),
            "day": np.asarray([34], dtype=np.int32),
            "capacity_utilisation": np.asarray([0.99], dtype=np.float32),
        },
        {
            "inventory": np.asarray([0, 5, 0], dtype=np.int32),
            "arrival_pipeline": np.asarray(
                [[0, 0, 60, 0], [0, 40, 0, 0], [20, 0, 0, 0]],
                dtype=np.int32,
            ),
            "demand_history": np.asarray(
                [
                    [42, 32, 48],
                    [39, 31, 50],
                    [45, 34, 55],
                    [47, 36, 53],
                    [41, 33, 51],
                    [44, 35, 56],
                    [48, 37, 58],
                ],
                dtype=np.int32,
            ),
            "day": np.asarray([47], dtype=np.int32),
            "capacity_utilisation": np.asarray([0.015], dtype=np.float32),
        },
    ]


def load_policy_module(path: Path) -> ModuleType:
    """Import a policy file under a unique module name."""
    module_name = f"student_policy_{uuid.uuid4().hex}"
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to create an import specification for {path}.")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def normalize_action(action: Any) -> tuple[int, int, int]:
    """Validate and normalize the returned action."""
    if isinstance(action, np.ndarray):
        action = action.tolist()
    if not isinstance(action, (list, tuple)):
        raise TypeError("run_policy() must return a list, tuple or NumPy array.")
    if len(action) != 3:
        raise ValueError("run_policy() must return exactly three order quantities.")

    normalized: list[int] = []
    for index, value in enumerate(action, start=1):
        if isinstance(value, (bool, np.bool_)):
            raise TypeError(f"Product {index} action must be an integer, not bool.")
        if not isinstance(value, (int, np.integer)):
            raise TypeError(f"Product {index} action must be an integer.")
        integer = int(value)
        if integer not in VALID_QUANTITIES:
            raise ValueError(
                f"Product {index} action {integer} is invalid; use 0, 10, ..., 100."
            )
        normalized.append(integer)
    return tuple(normalized)


def observations_equal(
    left: dict[str, np.ndarray], right: dict[str, np.ndarray]
) -> bool:
    return left.keys() == right.keys() and all(
        np.array_equal(left[key], right[key]) for key in left
    )


def find_forbidden_env_calls(path: Path) -> list[str]:
    """Detect direct calls to env.step() or env.reset() in executable code."""
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source, filename=str(path))
    findings: list[str] = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Attribute):
            continue
        if node.func.attr not in {"step", "reset"}:
            continue
        base = node.func.value
        if isinstance(base, ast.Name) and base.id == "env":
            findings.append(f"line {getattr(node, 'lineno', '?')}: env.{node.func.attr}()")
    return findings


def validate_signature(run_policy: Any) -> None:
    """Require the exact student-facing interface run_policy(observation)."""
    if not callable(run_policy):
        raise TypeError(
            "The policy file must define a callable function named "
            "run_policy(observation)."
        )

    signature = inspect.signature(run_policy)
    parameters = list(signature.parameters.values())

    if len(parameters) != 1:
        raise TypeError(
            "run_policy() must accept exactly one argument named observation."
        )

    parameter = parameters[0]
    if parameter.name != "observation":
        raise TypeError(
            "The argument of run_policy() must be named observation."
        )

    if parameter.kind not in (
        inspect.Parameter.POSITIONAL_ONLY,
        inspect.Parameter.POSITIONAL_OR_KEYWORD,
    ):
        raise TypeError(
            "run_policy(observation) must use a normal positional argument."
        )

    if parameter.default is not inspect.Parameter.empty:
        raise TypeError(
            "The observation argument must not have a default value."
        )


def run_validation(path: Path, max_seconds_per_call: float) -> list[str]:
    failures: list[str] = []

    try:
        forbidden = find_forbidden_env_calls(path)
        if forbidden:
            failures.append(
                "Direct environment calls are not permitted: " + ", ".join(forbidden)
            )
    except Exception as exc:  # syntax errors are also reported during import
        failures.append(f"Unable to inspect source code: {exc}")

    try:
        import_started = time.perf_counter()
        module = load_policy_module(path)
        import_seconds = time.perf_counter() - import_started
    except Exception as exc:
        failures.append(f"Policy file could not be imported: {type(exc).__name__}: {exc}")
        return failures

    try:
        run_policy = getattr(module, "run_policy")
        validate_signature(run_policy)
    except Exception as exc:
        failures.append(str(exc))
        return failures

    observations = representative_observations()

    # Interface, output and input-mutation checks.
    for index, observation in enumerate(observations, start=1):
        before = deepcopy(observation)
        try:
            action = normalize_action(run_policy(observation))
            _ = action
        except Exception as exc:
            failures.append(
                f"Representative observation {index} failed: {type(exc).__name__}: {exc}"
            )
            continue
        if not observations_equal(before, observation):
            failures.append(
                f"Representative observation {index}: run_policy() modified its input."
            )

    # Determinism is checked across fresh module loads. This permits a policy to
    # maintain deterministic internal state across calls, while ensuring that a
    # fresh policy load given the same observation sequence produces the same actions.
    try:
        sequences: list[list[tuple[int, int, int]]] = []
        for _ in range(3):
            fresh_module = load_policy_module(path)
            fresh_policy = getattr(fresh_module, "run_policy")
            sequence = [
                normalize_action(fresh_policy(deepcopy(observation)))
                for observation in observations
            ]
            sequences.append(sequence)
        if not all(sequence == sequences[0] for sequence in sequences[1:]):
            failures.append(
                "Determinism check failed: fresh policy loads produced different actions "
                "for the same observation sequence. Do not use uncontrolled randomness "
                "during evaluation."
            )
    except Exception as exc:
        failures.append(f"Determinism check failed with an error: {exc}")

    # Basic inference-time check. Import/model-loading time is reported separately.
    try:
        timing_module = load_policy_module(path)
        timing_policy = getattr(timing_module, "run_policy")
        repetitions = 100
        started = time.perf_counter()
        for iteration in range(repetitions):
            observation = deepcopy(observations[iteration % len(observations)])
            normalize_action(timing_policy(observation))
        elapsed = time.perf_counter() - started
        seconds_per_call = elapsed / repetitions
        if seconds_per_call > max_seconds_per_call:
            failures.append(
                f"Average inference time {seconds_per_call:.4f}s exceeds the local "
                f"limit of {max_seconds_per_call:.4f}s per call."
            )
    except Exception as exc:
        failures.append(f"Runtime check failed: {type(exc).__name__}: {exc}")
        seconds_per_call = float("nan")

    print(f"Policy file: {path}")
    print(f"Import/model-loading time: {import_seconds:.4f}s")
    if np.isfinite(seconds_per_call):
        print(f"Average inference time: {seconds_per_call:.6f}s per call")
    return failures


def main() -> int:
    parser = argparse.ArgumentParser(
        description=(
            "Validate a local inventory-project submission file that defines "
            "run_policy(observation)."
        )
    )
    parser.add_argument("policy_file", type=Path, help="Path to the policy .py file")
    parser.add_argument(
        "--max-seconds-per-call",
        type=float,
        default=0.25,
        help=(
            "Maximum allowed average inference time per call during this local "
            "check (default: 0.25 seconds)"
        ),
    )
    args = parser.parse_args()

    path = args.policy_file.resolve()
    if not path.is_file():
        print(f"FAIL: Policy file not found: {path}")
        return 1
    if path.suffix.lower() != ".py":
        print("FAIL: Policy submission must be a Python (.py) file.")
        return 1
    if args.max_seconds_per_call <= 0:
        print("FAIL: --max-seconds-per-call must be positive.")
        return 1

    failures = run_validation(path, args.max_seconds_per_call)
    if failures:
        print("\nVALIDATION FAILED")
        for number, failure in enumerate(failures, start=1):
            print(f"  {number}. {failure}")
        return 1

    print("\nVALIDATION PASSED")
    print(
        "The policy satisfies the local interface checks. Passing this script does "
        "not guarantee leaderboard eligibility, security compliance or performance."
    )
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 11. Final results across all trained techniques

Seven techniques were implemented and evaluated on the local harness (`evaluation.py` /
`src/evaluation/`) across three suites: Suite A (assigned configuration, 40 seeds x 5
scenario modes = 200 episodes), Suite B (7 legal configuration variants x 5 scenario
modes x 6 seeds = 210 episodes, robustness check), and Suite C (a locked, held-out proxy
of 125 episodes never used for hyperparameter tuning). The 5 scenario modes are the 4
declared demand families (`stationary`, `seasonal`, `trend`, `shock`) plus a local
`random`/`mixed` testing mode that samples combinations of the declared components --
not five separately-declared families. Results below come from
`eval_results/phase2/phase2_suite_summary.csv`, produced by `run_phase2_evaluation.py`.

In [ ]:
phase2_summary = pd.read_csv("eval_results/phase2/phase2_suite_summary.csv")
suite_a_summary = (
    phase2_summary[phase2_summary["suite"] == "A"]
    .sort_values("mean_cost")
    .loc[:, ["policy", "mean_cost", "std_cost", "p95_cost", "mean_service_level", "minimum_scenario_service"]]
    .reset_index(drop=True)
)
suite_a_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ordered = suite_a_summary.sort_values("mean_cost", ascending=False)
axes[0].barh(ordered["policy"], ordered["mean_cost"], xerr=ordered["std_cost"], color="#4c72b0")
axes[0].set_xlabel("Suite A mean episode cost (lower is better)")
axes[0].set_title("Cost comparison (error bars = std across seeds)")

axes[1].barh(ordered["policy"], ordered["mean_service_level"], color="#55a868")
axes[1].axvline(0.95, color="red", linestyle="--", linewidth=1, label="95% service threshold")
axes[1].set_xlabel("Suite A mean service level")
axes[1].set_title("Service level comparison")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 12. Champion selection (final 5-technique portfolio)

Seven implemented techniques were narrowed to five using a lexicographic tournament,
re-applied after a second improvement pass (see below):

1. **Reject invalid or non-deterministic policies** -- none rejected; all seven pass
   `policy_validation_tests.py` and the full audit in `src/audit/checks.py`.
2. **Reject service level below 95%** (an internal project selection rule adopted for
   this portfolio, not an official leaderboard qualification requirement) --
   REINFORCE (90.9%) is rejected here.
3. **Reject severe boundary failure** -- none of the remaining six show one.
4. **Minimize mean validation cost** (Suite A) -- ranks: PPO (91,470) < DQN (96,871) <
   Neural SARSA (111,731) < A2C (114,669) < TD(lambda) (122,769) < Double DQN (126,999).
5. **Top-5 cutoff**: TD(lambda) (122,769) vs Double DQN (126,999) is a clear ~3.4% gap --
   no tiebreak needed. TD(lambda) takes the 5th slot.

**Final portfolio: PPO, DQN, Neural SARSA, A2C, TD(lambda).**
Double DQN remains fully implemented, tested and evaluated, but is not part of the
final 5-technique submission this time.

### Two feature pipelines are in use, side by side

- **35-dim legacy pipeline** (`training_utils/obs_wrapper.py`, function
  `flatten_observation`): inventory, pipeline, 3/7-day demand means and std,
  day, capacity utilisation, inventory position, and a few derived ratios --
  hand-engineered once during iteration 1 and never changed since. Used by
  **PPO, DQN, A2C, and REINFORCE**.
- **Representation B, 76-dim** (`src/features/{observation,engineered}.py`):
  38 raw normalized features (Representation A) + 38 further-engineered
  features (per-product inventory-position gap, lead-time-demand estimate,
  days-of-supply, demand trend; plus 8 global capacity/horizon features).
  Built later (iteration 2) specifically for the newer custom algorithms and
  fixed for a zero-padding bug in iteration 3. Used by **Neural SARSA and
  Double DQN**.
- **TD(lambda)** uses neither -- it has its own 54-dim sparse, coarse-coded
  "band" discretization (`src/algorithms/common/discretizer.py`) purpose-built
  for its linear function approximator over a reduced 48-action catalogue.

These are three genuinely different, independently-verified pipelines (see
the exported-vs-training-side parity check for every technique) -- not one
canonical feature set reused everywhere.

Three notable engineering findings from the recovery/refinement phases:
- **PPO** was the worst iteration-1 technique (391,365 cost) and became one of the
  *best* after a 6-configuration entropy-coefficient x learning-rate screen (which
  showed learning rate, not entropy, was the deciding factor) followed by promoting
  the winning configuration to 1.5M timesteps -- final cost 91,470, better than DQN.
- **Neural SARSA** went through two rounds of work. Round 1 (single screening run)
  landed at 154,372 -- narrowly missing the original final 5. A follow-up audit found
  the engineered-feature pipeline had a real bug (early-episode demand means included
  zero-padded rows instead of masking them out, and "days of supply" used the wrong
  window). After fixing the features and running a proper 3-seed promotion of the
  winning screening config, Neural SARSA dropped to **111,731 cost / 99.3% service**
  -- the 3rd-best technique overall, and a clear top-5 qualifier.
- **Double DQN** was fixed twice. First, a real checkpoint-selection bug
  (`EvalCallback`'s first evaluation firing while `learning_starts` was still active)
  was found and fixed, improving it from 173,958 to 153,822. Later, a cost-source
  diagnosis showed it was still buying service via brute-force over-ordering (54%
  capacity utilisation vs DQN's 26%). Retraining on the corrected engineered features
  with a longer replay warm-up cut it further to 126,999 cost / 98.8% service and
  eliminated discard cost entirely -- a big improvement, but not quite enough to beat
  TD(lambda) for the final 5th slot this time.
- **A2C** refinement was attempted (12-configuration screen + 3-seed promotion of the
  top 2) but did not beat the original iteration-1 model, so the original was kept --
  not every refinement attempt succeeds, and the honest outcome is reported rather
  than forced.

## 12a. Reward shaping used during training

All five techniques in the final portfolio (PPO, DQN, Neural SARSA, A2C and
TD(lambda)) were trained with an auxiliary reward-shaping wrapper
(`training_utils/reward_shaping.py`'s `ShapedReward`), **not** the raw
environment reward alone. The wrapper adds three dense terms derived only
from the same `info` dict the environment already returns:

- `+ alpha * (service_level - 1.0)` -- an extra penalty for stockouts,
  proportional to how far service level falls short of 1.0.
- `- beta * max(0, capacity_utilisation - 0.85)` -- discourages pushing
  utilisation (and therefore discards) too close to capacity.
- `- gamma * n_orders_placed / 3` -- a gentle nudge toward batching orders
  rather than ordering every product every day.

All three coefficients are annealed **linearly to zero** over
`anneal_steps` environment steps (set to the full training budget for each
run), so by the end of training the effective reward converges back to the
unmodified environment reward. Every number reported anywhere in this
notebook (local Suite A/B/C validation, the comparison plots above, and the
leaderboard submissions themselves) is computed on the **raw, unshaped**
environment cost -- shaping is a training-time-only convenience signal and
never affects evaluation or the frozen `run_policy()` submissions.


## 12b. Learning curves and convergence evidence

Training-time convergence evidence for each of the five final techniques,
using the actual logs produced during screening/promotion (kept under
`eval_results/training_curves/`):

- **PPO / DQN / A2C** -- Stable-Baselines3's `EvalCallback` periodically
  evaluates the current policy during training; the stored
  `evaluations.npz` files give a genuine training-time learning curve
  (evaluation cost vs. training timesteps) for each technique's final
  promoted/champion run.
- **Neural SARSA** -- parsed directly from the promotion log
  (`neural_sarsa_promote.log`) for the deployed configuration
  (`sarsa_c`, seed `20260825`): evaluation cost drops sharply over the
  first ~100k transitions and then flattens out by the end of the
  500k-transition run.
- **TD(lambda)** -- a linear model trained with True Online SARSA(lambda)
  does not use a replay/checkpoint evaluation loop the same way, so no
  per-step curve was logged for the final run. Instead, two forms of
  convergence/robustness evidence are shown below: hyperparameter
  sensitivity from the grid search (cost vs. lambda, for each gamma), and
  cost stability across 3 independent seeds of the final promoted
  configuration (`eval_results/training_curves/td_lambda_*_results.json`).


In [ ]:
import json
import re

curves_dir = Path("eval_results/training_curves")

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# --- PPO / DQN / A2C: genuine SB3 EvalCallback training curves ---
sb3_curves = [
    ("PPO", "ppo_evaluations.npz", axes[0, 0]),
    ("DQN", "dqn_evaluations.npz", axes[0, 1]),
    ("A2C", "a2c_evaluations.npz", axes[0, 2]),
]
for name, fname, ax in sb3_curves:
    data = np.load(curves_dir / fname)
    timesteps = data["timesteps"]
    mean_reward = data["results"].mean(axis=1)
    mean_cost = -mean_reward * 100.0  # environment reward = -cost / 100
    ax.plot(timesteps, mean_cost, marker="o", markersize=3, color="#4c72b0")
    ax.set_title(f"{name}: eval cost vs training steps")
    ax.set_xlabel("Training timesteps")
    ax.set_ylabel("Mean eval cost")

# --- Neural SARSA: parsed from the promoted training log (sarsa_c, seed 20260825) ---
with open(curves_dir / "neural_sarsa_promote.log", "r", encoding="utf-16") as f:
    log_lines = f.read().splitlines()

run_id = "promoted_sarsa_c_seed20260825"
run_active = False
ns_steps, ns_costs = [], []
for line in log_lines:
    if line.startswith("==="):
        run_active = run_id in line
        continue
    if run_active:
        m = re.match(r"\[step\s+(\d+)\].*?eval_cost=([\d,]+\.\d+)", line)
        if m:
            ns_steps.append(int(m.group(1)))
            ns_costs.append(float(m.group(2).replace(",", "")))

ax = axes[1, 0]
ax.plot(ns_steps, ns_costs, marker="o", markersize=3, color="#c44e52")
ax.set_title("Neural SARSA: eval cost vs transitions\n(promoted sarsa_c, seed 20260825)")
ax.set_xlabel("Training transitions")
ax.set_ylabel("Mean eval cost")

# --- TD(lambda): hyperparameter sensitivity + 3-seed stability ---
with open(curves_dir / "td_lambda_grid_search_results.json") as f:
    grid = json.load(f)

ax = axes[1, 1]
for gamma_val in sorted({row["gamma"] for row in grid}):
    rows = sorted([r for r in grid if r["gamma"] == gamma_val], key=lambda r: r["lambda"])
    ax.plot(
        [r["lambda"] for r in rows],
        [r["best_eval_cost"] for r in rows],
        marker="o",
        label=f"gamma={gamma_val}",
    )
ax.set_title("TD(lambda): hyperparameter sensitivity")
ax.set_xlabel("lambda")
ax.set_ylabel("Best eval cost")
ax.legend(fontsize=8)

with open(curves_dir / "td_lambda_three_seed_results.json") as f:
    three_seed = json.load(f)

ax = axes[1, 2]
seeds = [str(r["seed"]) for r in three_seed]
seed_costs = [r["best_eval_cost"] for r in three_seed]
ax.bar(seeds, seed_costs, color="#8172b2")
ax.set_title("TD(lambda): stability across 3 seeds\n(final promoted configuration)")
ax.set_xlabel("Seed")
ax.set_ylabel("Best eval cost")

plt.tight_layout()
plt.show()


## 12c. Export steps used to produce each frozen submission

After training/promotion finished (in the main project repository, using the
scripts under `training_pipelines/`), each winning checkpoint was exported
**once** to the artifact frozen in `submissions/<technique>/`:

| Technique | Export call | Source checkpoint | Frozen artifact |
|---|---|---|---|
| PPO | `PPO.load(...)` model object's `.save(path)` | `models/ppo_v2/promoted_ent0.0_lr0.0003/best_model.zip` | `submissions/ppo/model.zip` |
| DQN | SB3 `.save(path)` | `models/dqn/best_model.zip` | `submissions/dqn/model.zip` |
| A2C | SB3 `.save(path)` | `models/a2c/best_model.zip` | `submissions/a2c/model.zip` |
| Neural SARSA | `torch.save(q_network.state_dict(), path)` | promoted config `sarsa_c`, seed `20260825` | `submissions/neural_sarsa/policy_state.pt` |
| TD(lambda) | `np.savez(path, weights=..., feature_bounds=...)` | 3-seed promoted run | `submissions/td_lambda/policy_weights.npz` |

Each `submissions/<technique>/policy.py` loads its artifact exactly once at
import time (module-level, outside `run_policy`) and never calls `.save()`,
`.learn()`, or any other mutating/training method -- the cell below loads
every frozen artifact directly from this package to verify each one is
present and in the expected format.


In [ ]:
import torch
from stable_baselines3 import A2C, DQN, PPO

sb3_artifacts = [
    ("PPO", PPO, "submissions/ppo/model.zip"),
    ("DQN", DQN, "submissions/dqn/model.zip"),
    ("A2C", A2C, "submissions/a2c/model.zip"),
]
for name, cls, path in sb3_artifacts:
    model = cls.load(path, device="cpu")
    print(f"{name:12s} loaded OK from {path} -- policy: {type(model.policy).__name__}")

sarsa_checkpoint = torch.load("submissions/neural_sarsa/policy_state.pt", map_location="cpu")
n_sarsa_params = sum(v.numel() for v in sarsa_checkpoint["model_state"].values())
print(
    f"{'Neural SARSA':12s} loaded OK -- obs_dim={sarsa_checkpoint['obs_dim']}, "
    f"n_actions={sarsa_checkpoint['n_actions']}, {n_sarsa_params:,} total parameters"
)

td_lambda_weights = np.load("submissions/td_lambda/policy_weights.npz")
print(f"{'TD(lambda)':12s} loaded OK -- arrays: {list(td_lambda_weights.keys())}")

for technique in ["ppo", "dqn", "neural_sarsa", "a2c", "td_lambda"]:
    with open(f"submissions/{technique}/metadata.json") as f:
        meta = json.load(f)
    print(f"{technique:14s} -> {meta['technique_category']}")


## 13. Final submission checks

Run `policy_validation_tests.py` against every policy file in the final portfolio before
freezing the submission.

In [ ]:
import subprocess

final_portfolio = ["ppo", "dqn", "neural_sarsa", "a2c", "td_lambda"]

for technique in final_portfolio:
    policy_path = f"submissions/{technique}/policy.py"
    result = subprocess.run(
        [sys.executable, "policy_validation_tests.py", policy_path],
        capture_output=True,
        text=True,
    )
    status = "PASS" if result.returncode == 0 else "FAIL"
    print(f"[{status}] {technique:12s} -> {policy_path}")
    if result.returncode != 0:
        print(result.stdout[-1000:])
        print(result.stderr[-1000:])